<a href="https://colab.research.google.com/github/Exponentik/AddressStandartization/blob/master/%D0%9E%D0%B1%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import zipfile
import shutil

import pandas as pd
import numpy as np
import pywt


# =========================
# НАСТРОЙКИ РАСПАКОВКИ
# =========================

zip_path = '/content/здоровые студенты.zip'
extract_dir = '/content/студенты_распаковано'
target_dir = '/content/логи_все_вместе_2'

os.makedirs(extract_dir, exist_ok=True)
os.makedirs(target_dir, exist_ok=True)


# =========================
# НАСТРОЙКИ ВЕЙВЛЕТ-ОБРАБОТКИ
# =========================

INPUT_PATTERN = "/content/логи_все_вместе_2/*.txt"

OUTPUT_PREFIX = "/content/data/wavelet_summary"

FILTERED_OUTPUT_DIR = "/content/data/wavelet_filtered_signals"
FILTERED_COMBINED_PATH = "/content/data/wavelet_filtered_all.csv"

WAVELET_NAME = "sym8"

# Нужные масштабы.
# Если в файле max_level >= 4, будут сохранены именно A4, D4, D3, D2, D1.
SCALES_TO_SAVE = ["A4", "D4", "D3", "D2", "D1"]

# Если True — для признаков var/std можно оставить полный max_level,
# как было в твоем коде.
# Если False — summary тоже будет считаться только по SCALES_TO_SAVE.
SUMMARY_USE_FULL_MAX_LEVEL = True


# =========================
# 1. РАСПАКОВКА И СБОР ЛОГОВ
# =========================

def try_fix_name(name):
    candidates = [
        ('cp437', 'utf-8'),
        ('latin1', 'utf-8'),
        ('cp866', 'utf-8'),
    ]

    for enc_from, enc_to in candidates:
        try:
            fixed = name.encode(enc_from).decode(enc_to)
            if fixed != name:
                return fixed
        except Exception:
            pass

    return name


def unpack_and_collect_logs():
    os.makedirs(extract_dir, exist_ok=True)
    os.makedirs(target_dir, exist_ok=True)

    # 1. Распаковываем архив
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)

    # 2. Исправляем имена файлов и папок снизу вверх
    for root, dirs, files in os.walk(extract_dir, topdown=False):
        for name in files:
            old_path = os.path.join(root, name)
            new_name = try_fix_name(name)
            new_path = os.path.join(root, new_name)

            if old_path != new_path and not os.path.exists(new_path):
                os.rename(old_path, new_path)

        for name in dirs:
            old_path = os.path.join(root, name)
            new_name = try_fix_name(name)
            new_path = os.path.join(root, new_name)

            if old_path != new_path and not os.path.exists(new_path):
                os.rename(old_path, new_path)

    # 3. Удаляем служебную папку macOS
    macosx_dir = os.path.join(extract_dir, '__MACOSX')
    if os.path.exists(macosx_dir):
        shutil.rmtree(macosx_dir)

    # 4. Ищем все папки "логи" и копируем файлы
    for root, dirs, files in os.walk(extract_dir):
        if os.path.basename(root).strip().lower() == 'логи':
            parent_name = os.path.basename(os.path.dirname(root))

            for filename in files:
                src = os.path.join(root, filename)

                if not os.path.isfile(src):
                    continue

                dst = os.path.join(target_dir, filename)

                # Если имя уже есть, добавляем имя родительской папки
                if os.path.exists(dst):
                    name, ext = os.path.splitext(filename)
                    dst = os.path.join(target_dir, f'{parent_name}_{name}{ext}')

                shutil.copy2(src, dst)
                print(f'Скопирован: {src} -> {dst}')

    print('\nРаспаковка и сбор логов завершены.')
    print('Итоговая папка:', target_dir)


# =========================
# 2. ЗАГРУЗКА ФАЙЛОВ
# =========================

def load_sensor_file(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    header_idx = None

    for i, line in enumerate(lines):
        if line.lower().startswith("milliseconds;"):
            header_idx = i
            break

    if header_idx is None:
        raise ValueError(f"Не найден заголовок в {path}")

    df = pd.read_csv(
        path,
        sep=';',
        skiprows=header_idx,
        engine='python',
        on_bad_lines='skip'
    )

    # Удаляем пустые колонки, которые могут появляться из-за ; в конце строки
    df = df.dropna(axis=1, how='all')

    return df


# =========================
# 3. ВЕЙВЛЕТ-ФУНКЦИИ
# =========================

def get_vector_defs():
    vector_defs = []

    for i in range(1, 5):
        vector_defs.append((f'acc{i}', [f'ax{i}', f'ay{i}', f'az{i}']))
        vector_defs.append((f'gyro{i}', [f'gx{i}', f'gy{i}', f'gz{i}']))

    return vector_defs


def get_vector_signal(df, cols, base):
    missing = [c for c in cols if c not in df.columns]

    if missing:
        raise ValueError(f"В файле {base} отсутствуют колонки: {missing}")

    xyz = df[cols].apply(pd.to_numeric, errors='coerce')
    xyz = xyz.dropna()

    if len(xyz) < 2:
        raise ValueError(f"В файле {base} слишком мало корректных данных для {cols}")

    signal = np.sqrt(
        xyz[cols[0]] ** 2 +
        xyz[cols[1]] ** 2 +
        xyz[cols[2]] ** 2
    ).to_numpy(dtype=float)

    return signal


def parse_scale(scale):
    """
    'A4' -> ('A', 4)
    'D3' -> ('D', 3)
    """

    kind = scale[0].upper()
    level = int(scale[1:])

    if kind not in ["A", "D"]:
        raise ValueError(f"Некорректный масштаб: {scale}")

    return kind, level


def reconstruct_component(signal, wavelet, decomposition_level, scale):
    """
    Восстанавливает отдельную компоненту сигнала.

    Для decomposition_level = 4:
        coeffs[0] = A4
        coeffs[1] = D4
        coeffs[2] = D3
        coeffs[3] = D2
        coeffs[4] = D1
    """

    kind, scale_level = parse_scale(scale)

    if scale_level > decomposition_level:
        raise ValueError(
            f"Нельзя восстановить {scale}, "
            f"потому что доступный уровень разложения = {decomposition_level}"
        )

    coeffs = pywt.wavedec(
        signal,
        wavelet,
        level=decomposition_level,
        mode='symmetric'
    )

    selected_coeffs = [np.zeros_like(c) for c in coeffs]

    if kind == "A":
        if scale_level != decomposition_level:
            raise ValueError(
                f"Для аппроксимации можно восстановить только A{decomposition_level}, "
                f"а запрошено {scale}"
            )

        selected_coeffs[0] = coeffs[0]

    elif kind == "D":
        coeff_index = decomposition_level - scale_level + 1
        selected_coeffs[coeff_index] = coeffs[coeff_index]

    reconstructed = pywt.waverec(
        selected_coeffs,
        wavelet,
        mode='symmetric'
    )

    # После обратного преобразования длина иногда может отличаться на 1-2 отсчета
    reconstructed = reconstructed[:len(signal)]

    return reconstructed


def make_labels_for_summary(max_level):
    return [f'A{max_level}'] + [f'D{lvl}' for lvl in range(max_level, 0, -1)]


def get_required_decomposition_level(max_level, scales_to_save):
    """
    Для сохранения A4, D4, D3, D2, D1 нужен decomposition_level = 4.
    Если max_level файла больше 4, мы всё равно делаем отдельное разложение уровня 4
    для сохранения именно этих масштабов.
    """

    required_levels = []

    for scale in scales_to_save:
        _, lvl = parse_scale(scale)
        required_levels.append(lvl)

    required_level = max(required_levels)

    if max_level < required_level:
        raise ValueError(
            f"Файл слишком короткий: max_level={max_level}, "
            f"а для {scales_to_save} нужен уровень {required_level}"
        )

    return required_level


# =========================
# 4. ОБРАБОТКА ОДНОГО ФАЙЛА
# =========================

def process_file(path: str):
    df = load_sensor_file(path)
    base = os.path.basename(path)

    vector_defs = get_vector_defs()

    wavelet = pywt.Wavelet(WAVELET_NAME)
    max_level = pywt.dwt_max_level(len(df), wavelet.dec_len)

    if max_level < 1:
        raise ValueError(f"Слишком мало данных для DWT в файле {base}")

    # Для summary можно использовать полный max_level, как в старом коде
    summary_level = max_level

    # Для сохранения фильтрованных сигналов используем уровень,
    # который нужен для заданных масштабов, например 4 для A4-D1
    filtered_level = get_required_decomposition_level(
        max_level=max_level,
        scales_to_save=SCALES_TO_SAVE
    )

    if SUMMARY_USE_FULL_MAX_LEVEL:
        summary_labels = make_labels_for_summary(summary_level)
    else:
        summary_labels = SCALES_TO_SAVE

    long_rows = []

    wide_row = {
        'file_name': base,
        'n_samples': len(df),
        'wavelet': WAVELET_NAME,
        'max_level': max_level
    }

    filtered_df = pd.DataFrame()
    filtered_df['file_name'] = [base] * len(df)

    if 'milliseconds' in df.columns:
        filtered_df['milliseconds'] = pd.to_numeric(
            df['milliseconds'],
            errors='coerce'
        ).values
    else:
        filtered_df['sample_index'] = np.arange(len(df))

    for vec_name, cols in vector_defs:
        signal = get_vector_signal(df, cols, base)

        # Если после dropna длина сигнала стала меньше исходного df,
        # то filtered_df тоже надо привести к этой длине.
        # На практике, если NaN мало или нет, длина совпадает.
        if len(signal) != len(filtered_df):
            min_len = min(len(signal), len(filtered_df))
            signal = signal[:min_len]
            filtered_df = filtered_df.iloc[:min_len].copy()

        # -------------------------
        # 4.1. Summary var/std
        # -------------------------
        if SUMMARY_USE_FULL_MAX_LEVEL:
            coeffs = pywt.wavedec(
                signal,
                wavelet,
                level=summary_level,
                mode='symmetric'
            )

            labels = make_labels_for_summary(summary_level)

            for idx, scale in enumerate(labels):
                c = np.asarray(coeffs[idx], dtype=float)

                row = {
                    'file_name': base,
                    'vector': vec_name,
                    'scale': scale,
                    'var': float(np.var(c, ddof=1)) if len(c) > 1 else 0.0,
                    'std': float(np.std(c, ddof=1)) if len(c) > 1 else 0.0
                }

                long_rows.append(row)

                wide_row[f'{vec_name}_{scale}_var'] = row['var']
                wide_row[f'{vec_name}_{scale}_std'] = row['std']

        else:
            for scale in SCALES_TO_SAVE:
                component = reconstruct_component(
                    signal=signal,
                    wavelet=wavelet,
                    decomposition_level=filtered_level,
                    scale=scale
                )

                row = {
                    'file_name': base,
                    'vector': vec_name,
                    'scale': scale,
                    'var': float(np.var(component, ddof=1)) if len(component) > 1 else 0.0,
                    'std': float(np.std(component, ddof=1)) if len(component) > 1 else 0.0
                }

                long_rows.append(row)

                wide_row[f'{vec_name}_{scale}_var'] = row['var']
                wide_row[f'{vec_name}_{scale}_std'] = row['std']

        # -------------------------
        # 4.2. Сохранение фильтрованных сигналов
        # -------------------------
        filtered_df[f'{vec_name}_raw_module'] = signal

        for scale in SCALES_TO_SAVE:
            component = reconstruct_component(
                signal=signal,
                wavelet=wavelet,
                decomposition_level=filtered_level,
                scale=scale
            )

            filtered_df[f'{vec_name}_{scale}'] = component

    long_df = pd.DataFrame(long_rows)
    wide_df = pd.DataFrame([wide_row])

    return long_df, wide_df, filtered_df


# =========================
# 5. ОБРАБОТКА ПАПКИ
# =========================

def process_folder(input_glob: str):
    os.makedirs(os.path.dirname(OUTPUT_PREFIX), exist_ok=True)
    os.makedirs(FILTERED_OUTPUT_DIR, exist_ok=True)

    long_parts = []
    wide_parts = []
    filtered_parts = []

    paths = sorted(glob.glob(input_glob))

    if not paths:
        raise FileNotFoundError(f'Нет файлов по шаблону: {input_glob}')

    for path in paths:
        try:
            long_df, wide_df, filtered_df = process_file(path)

            long_parts.append(long_df)
            wide_parts.append(wide_df)
            filtered_parts.append(filtered_df)

            base = os.path.basename(path)
            name, _ = os.path.splitext(base)

            filtered_path = os.path.join(
                FILTERED_OUTPUT_DIR,
                f'{name}_wavelet_filtered.csv'
            )

            filtered_df.to_csv(filtered_path, index=False)

            print(f'OK: {base}')
            print(f'  Фильтрованные сигналы сохранены: {filtered_path}')

        except Exception as e:
            print(f'SKIP: {os.path.basename(path)} -> {e}')

    if not long_parts:
        raise FileNotFoundError(f'Нет корректных файлов по шаблону: {input_glob}')

    long_all = pd.concat(long_parts, ignore_index=True)
    wide_all = pd.concat(wide_parts, ignore_index=True)
    filtered_all = pd.concat(filtered_parts, ignore_index=True)

    return long_all, wide_all, filtered_all


# =========================
# 6. MAIN
# =========================

def main():
    unpack_and_collect_logs()

    long_df, wide_df, filtered_all = process_folder(INPUT_PATTERN)

    long_csv_path = OUTPUT_PREFIX + '_long.csv'
    wide_csv_path = OUTPUT_PREFIX + '_wide.csv'
    xlsx_path = OUTPUT_PREFIX + '.xlsx'

    long_df.to_csv(long_csv_path, index=False)
    wide_df.to_csv(wide_csv_path, index=False)
    filtered_all.to_csv(FILTERED_COMBINED_PATH, index=False)

    with pd.ExcelWriter(xlsx_path, engine='xlsxwriter') as writer:
        long_df.to_excel(writer, sheet_name='LongSummary', index=False)
        wide_df.to_excel(writer, sheet_name='WideSummary', index=False)

    print('\nГотово.')
    print(f'Saved: {xlsx_path}')
    print(f'Saved: {long_csv_path}')
    print(f'Saved: {wide_csv_path}')
    print(f'Saved filtered combined: {FILTERED_COMBINED_PATH}')
    print(f'Saved filtered separate files in: {FILTERED_OUTPUT_DIR}')


if __name__ == '__main__':
    main()

Скопирован: /content/студенты_распаковано/╨╖╨┤╨╛╤Ç╨╛╨▓╤ï╨╡ ╤ü╤é╤â╨┤╨╡╨╜╤é╤ï/23. Корнилова/логи/kornilova31 2024-04-08_14_30_43146327.txt -> /content/логи_все_вместе_2/23. Корнилова_kornilova31 2024-04-08_14_30_43146327.txt
Скопирован: /content/студенты_распаковано/╨╖╨┤╨╛╤Ç╨╛╨▓╤ï╨╡ ╤ü╤é╤â╨┤╨╡╨╜╤é╤ï/23. Корнилова/логи/kornilova49 2024-04-08_14_56_48766076.txt -> /content/логи_все_вместе_2/23. Корнилова_kornilova49 2024-04-08_14_56_48766076.txt
Скопирован: /content/студенты_распаковано/╨╖╨┤╨╛╤Ç╨╛╨▓╤ï╨╡ ╤ü╤é╤â╨┤╨╡╨╜╤é╤ï/23. Корнилова/логи/kornilova62 2024-04-08_15_06_24576950.txt -> /content/логи_все_вместе_2/23. Корнилова_kornilova62 2024-04-08_15_06_24576950.txt
Скопирован: /content/студенты_распаковано/╨╖╨┤╨╛╤Ç╨╛╨▓╤ï╨╡ ╤ü╤é╤â╨┤╨╡╨╜╤é╤ï/23. Корнилова/логи/kornilova21 2024-04-08_14_26_57266971.txt -> /content/логи_все_вместе_2/23. Корнилова_kornilova21 2024-04-08_14_26_57266971.txt
Скопирован: /content/студенты_распаковано/╨╖╨┤╨╛╤Ç╨╛╨▓╤ï╨╡ ╤ü╤é╤â╨┤╨╡╨╜╤é╤ï/23. Корнилова/логи/kornilova44 2

/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 8 is too high: all coefficients will experience boundary effects.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 8 is too high: all coefficients will experience boundary effects.
  warnings.warn(


OK: 1. Маслова ЕС_maslovaES49 2023-12-15_15_13_43746034.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/1. Маслова ЕС_maslovaES49 2023-12-15_15_13_43746034_wavelet_filtered.csv
OK: 1. Маслова ЕС_maslovaES51 2023-12-15_15_16_19373128.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/1. Маслова ЕС_maslovaES51 2023-12-15_15_16_19373128_wavelet_filtered.csv
OK: 1. Маслова ЕС_maslovaES52 2023-12-15_15_18_09086494.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/1. Маслова ЕС_maslovaES52 2023-12-15_15_18_09086494_wavelet_filtered.csv
OK: 1. Маслова ЕС_maslovaES61 2023-12-15_15_23_20272894.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/1. Маслова ЕС_maslovaES61 2023-12-15_15_23_20272894_wavelet_filtered.csv
OK: 1. Маслова ЕС_maslovaES62 2023-12-15_15_23_56156148.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/1. Маслова ЕС_maslovaES62 2023-12-15_1

/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients w

OK: 20. Корнилов_kornilov12 2024-03-28_14_33_06332179.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/20. Корнилов_kornilov12 2024-03-28_14_33_06332179_wavelet_filtered.csv
OK: 20. Корнилов_kornilov21 2024-03-28_14_39_17496636.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/20. Корнилов_kornilov21 2024-03-28_14_39_17496636_wavelet_filtered.csv
OK: 20. Корнилов_kornilov22 2024-03-28_14_39_46088810.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/20. Корнилов_kornilov22 2024-03-28_14_39_46088810_wavelet_filtered.csv
OK: 20. Корнилов_kornilov31 2024-03-28_14_43_08588856.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/20. Корнилов_kornilov31 2024-03-28_14_43_08588856_wavelet_filtered.csv
OK: 20. Корнилов_kornilov32 2024-03-28_14_43_26598046.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/20. Корнилов_kornilov32 2024-03-28_14_43_26598046_wavele

...

OK: kornilov12 2024-03-28_14_33_06332179.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/kornilov12 2024-03-28_14_33_06332179_wavelet_filtered.csv
OK: kornilov21 2024-03-28_14_39_17496636.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/kornilov21 2024-03-28_14_39_17496636_wavelet_filtered.csv
OK: kornilov22 2024-03-28_14_39_46088810.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/kornilov22 2024-03-28_14_39_46088810_wavelet_filtered.csv
OK: kornilov31 2024-03-28_14_43_08588856.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/kornilov31 2024-03-28_14_43_08588856_wavelet_filtered.csv
OK: kornilov32 2024-03-28_14_43_26598046.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/kornilov32 2024-03-28_14_43_26598046_wavelet_filtered.csv
OK: kornilov41 2024-03-28_14_58_26234129.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_sign

/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients will experience boundary effects.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 6 is too high: all coefficients w

OK: loseva48 2024-05-18_08_14_15019976.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/loseva48 2024-05-18_08_14_15019976_wavelet_filtered.csv
OK: loseva49 2024-05-18_08_14_52905278.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/loseva49 2024-05-18_08_14_52905278_wavelet_filtered.csv
OK: loseva51 2024-05-18_08_16_05292312.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/loseva51 2024-05-18_08_16_05292312_wavelet_filtered.csv
OK: loseva52 2024-05-18_08_17_14306629.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/loseva52 2024-05-18_08_17_14306629_wavelet_filtered.csv
OK: loseva61 2024-05-18_08_21_11641308.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/loseva61 2024-05-18_08_21_11641308_wavelet_filtered.csv
OK: loseva62 2024-05-18_08_21_30207409.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/loseva62 2024-05-1

/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 8 is too high: all coefficients will experience boundary effects.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 8 is too high: all coefficients will experience boundary effects.
  warnings.warn(


OK: maslovaES49 2023-12-15_15_13_43746034.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/maslovaES49 2023-12-15_15_13_43746034_wavelet_filtered.csv
OK: maslovaES51 2023-12-15_15_16_19373128.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/maslovaES51 2023-12-15_15_16_19373128_wavelet_filtered.csv
OK: maslovaES52 2023-12-15_15_18_09086494.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/maslovaES52 2023-12-15_15_18_09086494_wavelet_filtered.csv
OK: maslovaES61 2023-12-15_15_23_20272894.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/maslovaES61 2023-12-15_15_23_20272894_wavelet_filtered.csv
OK: maslovaES62 2023-12-15_15_23_56156148.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_filtered_signals/maslovaES62 2023-12-15_15_23_56156148_wavelet_filtered.csv
OK: moshkinDA12 2023-12-17_10_44_50860949.txt
  Фильтрованные сигналы сохранены: /content/data/wavelet_fi

In [ ]:
import shutil
from google.colab import files

folder_path = '/content/data/wavelet_filtered_signals'
archive_path = '/content/wavelet_filtered_signals'

shutil.make_archive(
    base_name=archive_path,
    format='zip',
    root_dir=folder_path
)

files.download(archive_path + '.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import zipfile
import shutil

zip_path = '/content/здоровые студенты.zip'
extract_dir = '/content/студенты_распаковано'
target_dir = '/content/логи_все_вместе_2'

os.makedirs(extract_dir, exist_ok=True)
os.makedirs(target_dir, exist_ok=True)

# 1. Распаковываем архив
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

# 2. Функция исправления битой кодировки имён
def try_fix_name(name):
    candidates = [
        ('cp437', 'utf-8'),
        ('latin1', 'utf-8'),
        ('cp866', 'utf-8'),
    ]
    for enc_from, enc_to in candidates:
        try:
            fixed = name.encode(enc_from).decode(enc_to)
            # если после "исправления" строка стала более читаемой — используем её
            if fixed != name:
                return fixed
        except:
            pass
    return name

# 3. Рекурсивно исправляем имена файлов и папок снизу вверх
for root, dirs, files in os.walk(extract_dir, topdown=False):
    for name in files:
        old_path = os.path.join(root, name)
        new_name = try_fix_name(name)
        new_path = os.path.join(root, new_name)
        if old_path != new_path and not os.path.exists(new_path):
            os.rename(old_path, new_path)

    for name in dirs:
        old_path = os.path.join(root, name)
        new_name = try_fix_name(name)
        new_path = os.path.join(root, new_name)
        if old_path != new_path and not os.path.exists(new_path):
            os.rename(old_path, new_path)

# 4. Удаляем служебную папку macOS, если есть
macosx_dir = os.path.join(extract_dir, '__MACOSX')
if os.path.exists(macosx_dir):
    shutil.rmtree(macosx_dir)

# 5. Ищем все папки с именем "логи" и копируем из них файлы
for root, dirs, files in os.walk(extract_dir):
    if os.path.basename(root).strip().lower() == 'логи':
        parent_name = os.path.basename(os.path.dirname(root))
        for filename in files:
            src = os.path.join(root, filename)
            if os.path.isfile(src):
                dst = os.path.join(target_dir, filename)

                # если имя файла уже существует — добавляем имя родительской папки
                if os.path.exists(dst):
                    dst = os.path.join(target_dir, f'{filename}')

                shutil.copy2(src, dst)
                print(f'Скопирован: {src} -> {dst}')

print('\nГотово.')
print('Итоговая папка:', target_dir)

Скопирован: /content/студенты_распаковано/здоровые студенты/28. Каленова/логи/kalenova12 2024-04-14_09_10_20696854.txt -> /content/логи_все_вместе_2/kalenova12 2024-04-14_09_10_20696854.txt
Скопирован: /content/студенты_распаковано/здоровые студенты/28. Каленова/логи/kalenova48 2024-04-14_09_34_18249982.txt -> /content/логи_все_вместе_2/kalenova48 2024-04-14_09_34_18249982.txt
Скопирован: /content/студенты_распаковано/здоровые студенты/28. Каленова/логи/kalenova41 2024-04-14_09_29_54680443.txt -> /content/логи_все_вместе_2/kalenova41 2024-04-14_09_29_54680443.txt
Скопирован: /content/студенты_распаковано/здоровые студенты/28. Каленова/логи/kalenova52 2024-04-14_09_38_45371464.txt -> /content/логи_все_вместе_2/kalenova52 2024-04-14_09_38_45371464.txt
Скопирован: /content/студенты_распаковано/здоровые студенты/28. Каленова/логи/kalenova31 2024-04-14_09_18_00888850.txt -> /content/логи_все_вместе_2/kalenova31 2024-04-14_09_18_00888850.txt
Скопирован: /content/студенты_распаковано/здоровые

In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.3 MB/s eta 0:00:00


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import pywt


def load_sensor_file(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    header_idx = None
    for i, line in enumerate(lines):
        if line.lower().startswith("milliseconds;"):
            header_idx = i
            break

    if header_idx is None:
        raise ValueError(f"Не найден заголовок в {path}")

    df = pd.read_csv(
        path,
        sep=';',
        skiprows=header_idx,
        engine='python',
        on_bad_lines='skip'
    )

    return df


def process_file(path: str):
    df = load_sensor_file(path)
    base = os.path.basename(path)

    vector_defs = []
    for i in range(1, 5):
        vector_defs.append((f'acc{i}', [f'ax{i}', f'ay{i}', f'az{i}']))
        vector_defs.append((f'gyro{i}', [f'gx{i}', f'gy{i}', f'gz{i}']))

    wavelet = pywt.Wavelet('sym8')
    max_level = pywt.dwt_max_level(len(df), wavelet.dec_len)

    if max_level < 1:
        raise ValueError(f"Слишком мало данных для DWT в файле {base}")

    labels = [f'A{max_level}'] + [f'D{lvl}' for lvl in range(max_level, 0, -1)]

    long_rows = []
    wide_row = {
        'file_name': base,
        'n_samples': len(df),
        'wavelet': 'sym8',
        'max_level': max_level
    }

    for vec_name, cols in vector_defs:
        missing = [c for c in cols if c not in df.columns]
        if missing:
            raise ValueError(f"В файле {base} отсутствуют колонки: {missing}")

        xyz = df[cols].apply(pd.to_numeric, errors='coerce')
        xyz = xyz.dropna()

        if len(xyz) < 2:
            raise ValueError(f"В файле {base} слишком мало корректных данных для {vec_name}")

        signal = np.sqrt(
            xyz[cols[0]] ** 2 +
            xyz[cols[1]] ** 2 +
            xyz[cols[2]] ** 2
        ).to_numpy(dtype=float)

        coeffs = pywt.wavedec(
            signal,
            wavelet,
            level=max_level,
            mode='symmetric'
        )

        for idx, scale in enumerate(labels):
            c = np.asarray(coeffs[idx], dtype=float)

            row = {
                'file_name': base,
                'vector': vec_name,
                'scale': scale,
                'var': float(np.var(c, ddof=1)) if len(c) > 1 else 0.0,
                'std': float(np.std(c, ddof=1)) if len(c) > 1 else 0.0
            }

            long_rows.append(row)

            wide_row[f'{vec_name}_{scale}_var'] = row['var']
            wide_row[f'{vec_name}_{scale}_std'] = row['std']

    return pd.DataFrame(long_rows), pd.DataFrame([wide_row])


def process_folder(input_glob: str):
    long_parts = []
    wide_parts = []

    for path in sorted(glob.glob(input_glob)):
        try:
            long_df, wide_df = process_file(path)
            long_parts.append(long_df)
            wide_parts.append(wide_df)
            print(f'OK: {os.path.basename(path)}')
        except Exception as e:
            print(f'SKIP: {os.path.basename(path)} -> {e}')

    if not long_parts:
        raise FileNotFoundError(f'Нет корректных файлов по шаблону: {input_glob}')

    return pd.concat(long_parts, ignore_index=True), pd.concat(wide_parts, ignore_index=True)


def main():
    output_prefix = "/content/data/wavelet_summary"
    input_pattern = "/content/логи_все_вместе_2/*.txt"

    long_df, wide_df = process_folder(input_pattern)

    long_df.to_csv(output_prefix + '_long.csv', index=False)
    wide_df.to_csv(output_prefix + '_wide.csv', index=False)

    with pd.ExcelWriter(output_prefix + '.xlsx', engine='xlsxwriter') as writer:
        long_df.to_excel(writer, sheet_name='LongSummary', index=False)
        wide_df.to_excel(writer, sheet_name='WideSummary', index=False)

    print(f'Saved: {output_prefix}.xlsx')
    print(f'Saved: {output_prefix}_long.csv')
    print(f'Saved: {output_prefix}_wide.csv')


if __name__ == '__main__':
    main()

OK: BondarevaSI11 2024-05-14_14_40_37331603.txt
OK: BondarevaSI12 2024-05-14_14_41_21244788.txt
OK: BondarevaSI21 2024-05-14_14_46_43904177.txt
OK: BondarevaSI22 2024-05-14_14_47_16127144.txt
OK: BondarevaSI31 2024-05-14_14_49_59028654.txt
OK: BondarevaSI32 2024-05-14_14_50_20916348.txt
OK: BondarevaSI41 2024-05-14_15_05_04560811.txt
OK: BondarevaSI42 2024-05-14_15_05_37721012.txt
OK: BondarevaSI43 2024-05-14_15_06_17647229.txt
OK: BondarevaSI44 2024-05-14_15_07_26862652.txt
OK: BondarevaSI45 2024-05-14_15_07_58664325.txt
OK: BondarevaSI46 2024-05-14_15_09_01968384.txt
OK: BondarevaSI47 2024-05-14_15_10_01809501.txt
OK: BondarevaSI48 2024-05-14_15_10_42176936.txt
OK: BondarevaSI49 2024-05-14_15_11_25723069.txt
OK: BondarevaSI51 2024-05-14_15_12_56080323.txt
OK: BondarevaSI52 2024-05-14_15_14_17167320.txt
OK: BondarevaSI61 2024-05-14_15_18_27572429.txt
OK: BondarevaSI62 2024-05-14_15_18_46877790.txt
OK: RyabininUV11 2024-02-02_14_32_54528275.txt
OK: RyabininUV12 2024-02-02_14_33_2803823

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score


# =========================
# НАСТРОЙКИ
# =========================
WIDE_CSV_PATH = '/content/data/wavelet_summary_wide.csv'
OUTPUT_PATH = '/content/data/k_summary_metrics.xlsx'

DROP_META_COLS = ['file_name', 'wavelet', 'n_samples', 'max_level']
USE_ONLY_COMMON_SCALES = True
COMMON_SCALES = ['A4', 'D4', 'D3', 'D2', 'D1']
MAX_NAN_RATIO = 0.40
N_COMPONENTS = 10
K_VALUES = list(range(2, 11))
RANDOM_STATE = 42


def is_common_scale_column(col_name, scales):
    return any(f'_{scale}_' in col_name for scale in scales)


def prepare_features(path):
    df = pd.read_csv(path)

    feature_cols = [c for c in df.columns if c not in DROP_META_COLS]
    if USE_ONLY_COMMON_SCALES:
        feature_cols = [c for c in feature_cols if is_common_scale_column(c, COMMON_SCALES)]

    X = df[feature_cols].copy()

    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce')

    nan_ratio = X.isna().mean()
    X = X.loc[:, nan_ratio <= MAX_NAN_RATIO]

    nunique = X.nunique(dropna=True)
    X = X.loc[:, nunique > 1]

    imputer = SimpleImputer(strategy='median')
    X_imp = imputer.fit_transform(X)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)

    pca_n = min(N_COMPONENTS, X_scaled.shape[0], X_scaled.shape[1])
    if pca_n < 2:
        pca_n = 2

    pca = PCA(n_components=pca_n, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_scaled)

    return df, X_pca


def main():
    df_raw, X_pca = prepare_features(WIDE_CSV_PATH)

    rows = []

    for k in K_VALUES:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        labels = model.fit_predict(X_pca)

        sil = silhouette_score(X_pca, labels)
        ch = calinski_harabasz_score(X_pca, labels)
        db = davies_bouldin_score(X_pca, labels)
        inertia = model.inertia_

        cluster_sizes = pd.Series(labels).value_counts().sort_index()
        min_size = int(cluster_sizes.min())
        max_size = int(cluster_sizes.max())
        mean_size = float(cluster_sizes.mean())
        std_size = float(cluster_sizes.std(ddof=0))

        rows.append({
            'k': k,
            'silhouette': sil,
            'calinski_harabasz': ch,
            'davies_bouldin': db,
            'inertia': inertia,
            'n_clusters_found': len(cluster_sizes),
            'min_cluster_size': min_size,
            'max_cluster_size': max_size,
            'mean_cluster_size': mean_size,
            'std_cluster_size': std_size
        })

    results = pd.DataFrame(rows)

    # нормированные ранги, чтобы выбрать "лучший k" по нескольким метрикам
    results['rank_silhouette'] = results['silhouette'].rank(ascending=False, method='min')
    results['rank_ch'] = results['calinski_harabasz'].rank(ascending=False, method='min')
    results['rank_db'] = results['davies_bouldin'].rank(ascending=True, method='min')
    results['rank_sum'] = results['rank_silhouette'] + results['rank_ch'] + results['rank_db']

    results = results.sort_values('k')

    best_by_rank = results.loc[results['rank_sum'].idxmin(), 'k']
    print('Best k by combined rank:', best_by_rank)
    print(results)

    with pd.ExcelWriter(OUTPUT_PATH, engine='xlsxwriter') as writer:
        results.to_excel(writer, sheet_name='Summary', index=False)

    print(f'Saved: {OUTPUT_PATH}')


if __name__ == '__main__':
    main()

Best k by combined rank: 3
    k  silhouette  calinski_harabasz  davies_bouldin       inertia  \
0   2    0.546707         427.184812        0.891911  16734.892982   
1   3    0.548546         302.689581        0.624374  14077.639348   
2   4    0.474777         283.005781        1.030655  11562.213750   
3   5    0.480826         270.273722        0.987219   9874.511214   
4   6    0.487458         255.716719        0.933111   8778.242970   
5   7    0.491234         251.066705        0.842020   7779.697618   
6   8    0.406932         242.462958        0.816285   7098.965206   
7   9    0.417050         250.844591        0.893578   6218.758106   
8  10    0.404516         249.020306        0.828464   5681.049233   

   n_clusters_found  min_cluster_size  max_cluster_size  mean_cluster_size  \
0                 2               155               370         262.500000   
1                 3                 1               370         175.000000   
2                 4                 1 

In [ ]:
def compute_elbow_scores(X_pca, k_range):
    rows = []

    for k in k_range:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        model.fit(X_pca)

        rows.append({
            'k': k,
            'inertia': model.inertia_
        })

    elbow_df = pd.DataFrame(rows)
    return elbow_df

def save_elbow_plot(elbow_df: pd.DataFrame, out_path: str):
    plt.figure(figsize=(8, 5))
    plt.plot(elbow_df['k'], elbow_df['inertia'], marker='o')
    plt.xlabel('Number of clusters k')
    plt.ylabel('Inertia')
    plt.title('Elbow method for KMeans')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score


# =========================
# НАСТРОЙКИ
# =========================
WIDE_CSV_PATH = '/content/data/wavelet_summary_wide.csv'
OUTPUT_DIR = '/content/data/clustering_results'

USE_ONLY_COMMON_SCALES = True
COMMON_SCALES = ['A4', 'D4', 'D3', 'D2', 'D1']

DROP_META_COLS = ['file_name', 'wavelet', 'n_samples', 'max_level']
MAX_NAN_RATIO = 0.40
K_RANGE = range(10, 11)
N_COMPONENTS = 10
RANDOM_STATE = 42
CLUSTER_METHOD = 'kmeans'


# =========================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =========================
def ensure_output_dir(path: str):
    os.makedirs(path, exist_ok=True)


def is_common_scale_column(col_name: str, scales):
    for scale in scales:
        if f'_{scale}_' in col_name:
            return True
    return False


def select_feature_columns(df: pd.DataFrame):
    feature_cols = [c for c in df.columns if c not in DROP_META_COLS]

    if USE_ONLY_COMMON_SCALES:
        feature_cols = [c for c in feature_cols if is_common_scale_column(c, COMMON_SCALES)]

    return feature_cols


def prepare_features(wide_csv_path: str):
    df = pd.read_csv(wide_csv_path)

    if 'file_name' not in df.columns:
        raise ValueError('В wide CSV нет колонки file_name')

    feature_cols = select_feature_columns(df)
    if not feature_cols:
        raise ValueError('Не удалось выбрать признаки для кластеризации')

    X = df[feature_cols].copy()

    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')

    nan_ratio = X.isna().mean()
    keep_nan_cols = nan_ratio[nan_ratio <= MAX_NAN_RATIO].index.tolist()
    X = X[keep_nan_cols]

    if X.shape[1] == 0:
        raise ValueError('После удаления колонок с большим числом NaN не осталось признаков')

    nunique = X.nunique(dropna=True)
    keep_var_cols = nunique[nunique > 1].index.tolist()
    X = X[keep_var_cols]

    if X.shape[1] == 0:
        raise ValueError('После удаления константных признаков не осталось признаков')

    imputer = SimpleImputer(strategy='median')
    X_imp = imputer.fit_transform(X)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)

    final_feature_cols = X.columns.tolist()

    info = {
        'n_files': len(df),
        'n_features_before': len(feature_cols),
        'n_features_after_nan_filter': len(keep_nan_cols),
        'n_features_final': len(final_feature_cols),
        'final_feature_cols': final_feature_cols
    }

    return df, X, X_scaled, final_feature_cols, info


def fit_pca(X_scaled, n_components=10):
    n_components_eff = min(n_components, X_scaled.shape[0], X_scaled.shape[1])
    if n_components_eff < 2:
        n_components_eff = min(2, X_scaled.shape[1], X_scaled.shape[0])

    pca = PCA(n_components=n_components_eff, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_scaled)
    return pca, X_pca


def find_best_k_kmeans(X_pca, k_range):
    rows = []

    for k in k_range:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        labels = model.fit_predict(X_pca)

        if k == 1 or len(set(labels)) < 2:
            sil = np.nan
        else:
            sil = silhouette_score(X_pca, labels)

        rows.append({'k': k, 'silhouette': sil})

    scores_df = pd.DataFrame(rows)
    valid_scores = scores_df.dropna(subset=['silhouette'])
    best_k = int(valid_scores.loc[valid_scores['silhouette'].idxmax(), 'k'])

    return best_k, scores_df


def compute_elbow_scores(X_pca, k_range):
    rows = []

    for k in k_range:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        model.fit(X_pca)

        rows.append({
            'k': k,
            'inertia': model.inertia_
        })

    elbow_df = pd.DataFrame(rows)
    return elbow_df


def run_kmeans(X_pca, k):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = model.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels) if len(set(labels)) > 1 else np.nan
    return labels, sil, model


def run_agglomerative(X_pca, k):
    model = AgglomerativeClustering(n_clusters=k)
    labels = model.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels) if len(set(labels)) > 1 else np.nan
    return labels, sil, model


def build_cluster_profiles(df_raw: pd.DataFrame, X_numeric: pd.DataFrame, labels):
    tmp = pd.concat(
        [
            df_raw[['file_name']].reset_index(drop=True),
            X_numeric.reset_index(drop=True)
        ],
        axis=1
    )
    tmp['cluster'] = labels

    profile_mean = tmp.groupby('cluster').mean(numeric_only=True)
    profile_median = tmp.groupby('cluster').median(numeric_only=True)
    cluster_sizes = tmp.groupby('cluster').size().rename('n_files').to_frame()

    return tmp, profile_mean, profile_median, cluster_sizes


def compute_top_features_by_cluster(profile_mean: pd.DataFrame):
    global_mean = profile_mean.mean(axis=0)
    diff = (profile_mean - global_mean).abs()
    importance = diff.mean(axis=0).sort_values(ascending=False)

    top_df = importance.reset_index()
    top_df.columns = ['feature', 'importance']
    return top_df


def save_pca_plot(result_df: pd.DataFrame, out_path: str):
    plt.figure(figsize=(10, 7))

    clusters = sorted(result_df['cluster'].unique())
    for cluster_id in clusters:
        part = result_df[result_df['cluster'] == cluster_id]
        plt.scatter(part['pca1'], part['pca2'], label=f'cluster {cluster_id}', s=40)

    plt.xlabel('PCA1')
    plt.ylabel('PCA2')
    plt.title('Clusters of behavior patterns')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def save_elbow_plot(elbow_df: pd.DataFrame, out_path: str):
    plt.figure(figsize=(8, 5))
    plt.plot(elbow_df['k'], elbow_df['inertia'], marker='o')
    plt.xlabel('Number of clusters k')
    plt.ylabel('Inertia')
    plt.title('Elbow method for KMeans')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def save_all_results(
    output_dir,
    result_df,
    scores_df,
    elbow_df,
    profile_mean,
    profile_median,
    cluster_sizes,
    top_features_df,
    pca,
    info
):
    ensure_output_dir(output_dir)

    result_csv = os.path.join(output_dir, 'file_clusters.csv')
    scores_csv = os.path.join(output_dir, 'kmeans_silhouette_scores.csv')
    elbow_csv = os.path.join(output_dir, 'kmeans_elbow_scores.csv')
    profile_mean_csv = os.path.join(output_dir, 'cluster_profiles_mean.csv')
    profile_median_csv = os.path.join(output_dir, 'cluster_profiles_median.csv')
    cluster_sizes_csv = os.path.join(output_dir, 'cluster_sizes.csv')
    top_features_csv = os.path.join(output_dir, 'top_cluster_features.csv')
    pca_plot_png = os.path.join(output_dir, 'clusters_pca.png')
    elbow_plot_png = os.path.join(output_dir, 'kmeans_elbow_plot.png')
    excel_path = os.path.join(output_dir, 'clustering_summary.xlsx')
    info_txt = os.path.join(output_dir, 'clustering_info.txt')

    result_df.to_csv(result_csv, index=False)
    scores_df.to_csv(scores_csv, index=False)
    elbow_df.to_csv(elbow_csv, index=False)
    profile_mean.to_csv(profile_mean_csv)
    profile_median.to_csv(profile_median_csv)
    cluster_sizes.to_csv(cluster_sizes_csv)
    top_features_df.to_csv(top_features_csv, index=False)
    save_pca_plot(result_df, pca_plot_png)
    save_elbow_plot(elbow_df, elbow_plot_png)

    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        result_df.to_excel(writer, sheet_name='FileClusters', index=False)
        scores_df.to_excel(writer, sheet_name='SilhouetteScores', index=False)
        elbow_df.to_excel(writer, sheet_name='ElbowScores', index=False)
        profile_mean.to_excel(writer, sheet_name='ClusterMean')
        profile_median.to_excel(writer, sheet_name='ClusterMedian')
        cluster_sizes.to_excel(writer, sheet_name='ClusterSizes')
        top_features_df.to_excel(writer, sheet_name='TopFeatures', index=False)

        pca_info_df = pd.DataFrame({
            'component': [f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))],
            'explained_variance_ratio': pca.explained_variance_ratio_
        })
        pca_info_df.to_excel(writer, sheet_name='PCAInfo', index=False)

    with open(info_txt, 'w', encoding='utf-8') as f:
        f.write('Clustering info\n')
        f.write('====================\n')
        for k, v in info.items():
            if k == 'final_feature_cols':
                f.write(f'{k}: {len(v)} features\n')
            else:
                f.write(f'{k}: {v}\n')

    print(f'Saved: {result_csv}')
    print(f'Saved: {scores_csv}')
    print(f'Saved: {elbow_csv}')
    print(f'Saved: {profile_mean_csv}')
    print(f'Saved: {profile_median_csv}')
    print(f'Saved: {cluster_sizes_csv}')
    print(f'Saved: {top_features_csv}')
    print(f'Saved: {pca_plot_png}')
    print(f'Saved: {elbow_plot_png}')
    print(f'Saved: {excel_path}')
    print(f'Saved: {info_txt}')


def main():
    ensure_output_dir(OUTPUT_DIR)

    df_raw, X_numeric, X_scaled, feature_cols, info = prepare_features(WIDE_CSV_PATH)

    print('Feature preparation:')
    print(f"Files: {info['n_files']}")
    print(f"Features before filtering: {info['n_features_before']}")
    print(f"Features after NaN filtering: {info['n_features_after_nan_filter']}")
    print(f"Final features: {info['n_features_final']}")

    pca, X_pca = fit_pca(X_scaled, n_components=N_COMPONENTS)

    best_k, scores_df = find_best_k_kmeans(X_pca, K_RANGE)
    elbow_df = compute_elbow_scores(X_pca, K_RANGE)

    print('\nSilhouette scores:')
    print(scores_df)
    print('Best k by silhouette:', best_k)

    print('\nElbow scores:')
    print(elbow_df)

    if CLUSTER_METHOD == 'kmeans':
        labels, sil, _ = run_kmeans(X_pca, best_k)
    elif CLUSTER_METHOD == 'agglomerative':
        labels, sil, _ = run_agglomerative(X_pca, best_k)
    else:
        raise ValueError("CLUSTER_METHOD должен быть 'kmeans' или 'agglomerative'")

    result_df = df_raw[['file_name']].copy()
    result_df['cluster'] = labels
    result_df['pca1'] = X_pca[:, 0]
    result_df['pca2'] = X_pca[:, 1] if X_pca.shape[1] > 1 else 0.0

    if 'n_samples' in df_raw.columns:
        result_df['n_samples'] = df_raw['n_samples']
    if 'max_level' in df_raw.columns:
        result_df['max_level'] = df_raw['max_level']

    print('\nFinal silhouette:', sil)
    print(result_df.head())

    merged_df, profile_mean, profile_median, cluster_sizes = build_cluster_profiles(df_raw, X_numeric, labels)
    top_features_df = compute_top_features_by_cluster(profile_mean)

    print('\nCluster sizes:')
    print(cluster_sizes)

    print('\nTop 20 most important features:')
    print(top_features_df.head(20))

    save_all_results(
        output_dir=OUTPUT_DIR,
        result_df=result_df,
        scores_df=scores_df,
        elbow_df=elbow_df,
        profile_mean=profile_mean,
        profile_median=profile_median,
        cluster_sizes=cluster_sizes,
        top_features_df=top_features_df,
        pca=pca,
        info=info
    )


if __name__ == '__main__':
    main()

Feature preparation:
Files: 525
Features before filtering: 80
Features after NaN filtering: 64
Final features: 64

Silhouette scores:
    k  silhouette
0  10    0.404516
Best k by silhouette: 10

Elbow scores:
    k      inertia
0  10  5681.049233

Final silhouette: 0.4045159544783605
                                     file_name  cluster      pca1      pca2  \
0  BondarevaSI11 2024-05-14_14_40_37331603.txt        3  5.224304 -1.407068   
1  BondarevaSI12 2024-05-14_14_41_21244788.txt        2  4.959378 -2.109583   
2  BondarevaSI21 2024-05-14_14_46_43904177.txt        0 -2.226149 -0.503596   
3  BondarevaSI22 2024-05-14_14_47_16127144.txt        4  0.211083  0.611633   
4  BondarevaSI31 2024-05-14_14_49_59028654.txt        3  6.247630 -2.321502   

   n_samples  max_level  
0        332          4  
1        416          4  
2        403          4  
3        488          5  
4        328          4  

Cluster sizes:
         n_files
cluster         
0            267
1             25

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.cluster import (
    KMeans,
    MiniBatchKMeans,
    AgglomerativeClustering,
    Birch,
    SpectralClustering,
    DBSCAN,
    OPTICS,
)
from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)

warnings.filterwarnings("ignore")


# =========================
# НАСТРОЙКИ
# =========================
WIDE_CSV_PATH = "/content/data/wavelet_summary_wide.csv"
OUTPUT_DIR = "/content/data/multi_clustering_results"

USE_ONLY_COMMON_SCALES = True
COMMON_SCALES = ["A4", "D4", "D3", "D2", "D1"]

DROP_META_COLS = ["file_name", "wavelet", "n_samples", "max_level"]
MAX_NAN_RATIO = 0.40
N_COMPONENTS = 10
RANDOM_STATE = 42

# для методов, которым нужно число кластеров
K_CANDIDATES = [8]


# =========================
# ПОДГОТОВКА
# =========================
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def is_common_scale_column(col_name: str, scales):
    return any(f"_{scale}_" in col_name for scale in scales)


def select_feature_columns(df: pd.DataFrame):
    feature_cols = [c for c in df.columns if c not in DROP_META_COLS]
    if USE_ONLY_COMMON_SCALES:
        feature_cols = [c for c in feature_cols if is_common_scale_column(c, COMMON_SCALES)]
    return feature_cols


def prepare_features(wide_csv_path: str):
    df = pd.read_csv(wide_csv_path)

    if "file_name" not in df.columns:
        raise ValueError("В wide CSV нет колонки file_name")

    feature_cols = select_feature_columns(df)
    if not feature_cols:
        raise ValueError("Не удалось выбрать признаки")

    X = df[feature_cols].copy()

    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    nan_ratio = X.isna().mean()
    keep_nan_cols = nan_ratio[nan_ratio <= MAX_NAN_RATIO].index.tolist()
    X = X[keep_nan_cols]

    if X.shape[1] == 0:
        raise ValueError("После удаления колонок с большим числом NaN не осталось признаков")

    nunique = X.nunique(dropna=True)
    keep_var_cols = nunique[nunique > 1].index.tolist()
    X = X[keep_var_cols]

    if X.shape[1] == 0:
        raise ValueError("После удаления константных признаков не осталось признаков")

    imputer = SimpleImputer(strategy="median")
    X_imp = imputer.fit_transform(X)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)

    pca_n = min(N_COMPONENTS, X_scaled.shape[0], X_scaled.shape[1])
    if pca_n < 2:
        pca_n = 2

    pca = PCA(n_components=pca_n, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_scaled)

    info = {
        "n_files": len(df),
        "n_features_final": X.shape[1],
        "pca_components": pca_n,
    }

    return df, X, X_scaled, X_pca, pca, info


# =========================
# ОЦЕНКА
# =========================
def safe_cluster_metrics(X, labels):
    labels = np.asarray(labels)
    unique = np.unique(labels)

    # для DBSCAN/OPTICS шум = -1, если все в шуме или один кластер — метрики не считаем
    non_noise = labels != -1
    if non_noise.sum() < 2:
        return np.nan, np.nan, np.nan, 0, int((labels == -1).sum())

    labels_nn = labels[non_noise]
    X_nn = X[non_noise]

    unique_nn = np.unique(labels_nn)
    n_clusters = len(unique_nn)
    n_noise = int((labels == -1).sum())

    if n_clusters < 2:
        return np.nan, np.nan, np.nan, n_clusters, n_noise

    sil = silhouette_score(X_nn, labels_nn)
    ch = calinski_harabasz_score(X_nn, labels_nn)
    db = davies_bouldin_score(X_nn, labels_nn)

    return sil, ch, db, n_clusters, n_noise


# =========================
# МОДЕЛИ
# =========================
def fit_predict_model(method_name, X_pca, k=None):
    if method_name == "KMeans":
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        labels = model.fit_predict(X_pca)

    elif method_name == "MiniBatchKMeans":
        model = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20, batch_size=64)
        labels = model.fit_predict(X_pca)

    elif method_name == "Agglomerative_Ward":
        model = AgglomerativeClustering(n_clusters=k, linkage="ward")
        labels = model.fit_predict(X_pca)

    elif method_name == "Agglomerative_Complete":
        model = AgglomerativeClustering(n_clusters=k, linkage="complete")
        labels = model.fit_predict(X_pca)

    elif method_name == "Agglomerative_Average":
        model = AgglomerativeClustering(n_clusters=k, linkage="average")
        labels = model.fit_predict(X_pca)

    elif method_name == "Birch":
        model = Birch(n_clusters=k)
        labels = model.fit_predict(X_pca)

    elif method_name == "SpectralClustering":
        model = SpectralClustering(
            n_clusters=k,
            affinity="nearest_neighbors",
            random_state=RANDOM_STATE,
            assign_labels="kmeans",
        )
        labels = model.fit_predict(X_pca)

    elif method_name == "GaussianMixture":
        model = GaussianMixture(n_components=k, random_state=RANDOM_STATE)
        labels = model.fit_predict(X_pca)

    elif method_name == "DBSCAN":
        model = DBSCAN(eps=1.6, min_samples=8)
        labels = model.fit_predict(X_pca)

    elif method_name == "OPTICS":
        model = OPTICS(min_samples=8, xi=0.05, min_cluster_size=0.03)
        labels = model.fit_predict(X_pca)

    else:
        raise ValueError(f"Неизвестный метод: {method_name}")

    return labels


def methods_with_fixed_k():
    return [
        "KMeans",
        "MiniBatchKMeans",
        "Agglomerative_Ward",
        "Agglomerative_Complete",
        "Agglomerative_Average",
        "Birch",
        "SpectralClustering",
        "GaussianMixture",
    ]


def methods_without_k():
    return [
        "DBSCAN",
        "OPTICS",
    ]


# =========================
# ПОИСК ЛУЧШЕГО K
# =========================
def evaluate_method_over_k(method_name, X_pca, k_candidates):
    rows = []
    best = None
    best_score = -np.inf

    for k in k_candidates:
        try:
            labels = fit_predict_model(method_name, X_pca, k=k)
            sil, ch, db, n_clusters, n_noise = safe_cluster_metrics(X_pca, labels)

            row = {
                "method": method_name,
                "k": k,
                "silhouette": sil,
                "calinski_harabasz": ch,
                "davies_bouldin": db,
                "n_clusters_found": n_clusters,
                "n_noise": n_noise,
            }
            rows.append(row)

            if pd.notna(sil) and sil > best_score:
                best_score = sil
                best = {
                    "method": method_name,
                    "best_k": k,
                    "labels": labels,
                    "silhouette": sil,
                    "calinski_harabasz": ch,
                    "davies_bouldin": db,
                    "n_clusters_found": n_clusters,
                    "n_noise": n_noise,
                }

        except Exception as e:
            rows.append({
                "method": method_name,
                "k": k,
                "silhouette": np.nan,
                "calinski_harabasz": np.nan,
                "davies_bouldin": np.nan,
                "n_clusters_found": np.nan,
                "n_noise": np.nan,
                "error": str(e),
            })

    return pd.DataFrame(rows), best


def evaluate_density_method(method_name, X_pca):
    try:
        labels = fit_predict_model(method_name, X_pca, k=None)
        sil, ch, db, n_clusters, n_noise = safe_cluster_metrics(X_pca, labels)

        result = {
            "method": method_name,
            "best_k": np.nan,
            "labels": labels,
            "silhouette": sil,
            "calinski_harabasz": ch,
            "davies_bouldin": db,
            "n_clusters_found": n_clusters,
            "n_noise": n_noise,
        }
        row = pd.DataFrame([{
            "method": method_name,
            "k": np.nan,
            "silhouette": sil,
            "calinski_harabasz": ch,
            "davies_bouldin": db,
            "n_clusters_found": n_clusters,
            "n_noise": n_noise,
        }])
        return row, result

    except Exception as e:
        row = pd.DataFrame([{
            "method": method_name,
            "k": np.nan,
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "n_clusters_found": np.nan,
            "n_noise": np.nan,
            "error": str(e),
        }])
        return row, None


# =========================
# ВИЗУАЛИЗАЦИИ И СОХРАНЕНИЕ
# =========================
def save_method_scatter(df_files, labels, method_name, out_dir):
    plot_df = df_files[["file_name"]].copy()
    plot_df["cluster"] = labels
    plot_df["pca1"] = df_files["pca1"]
    plot_df["pca2"] = df_files["pca2"]

    plt.figure(figsize=(8, 6))
    for cl in sorted(pd.unique(plot_df["cluster"])):
        part = plot_df[plot_df["cluster"] == cl]
        plt.scatter(part["pca1"], part["pca2"], label=f"cluster {cl}", s=20)

    plt.xlabel("PCA1")
    plt.ylabel("PCA2")
    plt.title(method_name)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{method_name}_pca.png"), dpi=150)
    plt.close()


def save_summary_bar(summary_df, out_path):
    valid = summary_df.dropna(subset=["silhouette"]).sort_values("silhouette", ascending=False)

    plt.figure(figsize=(12, 6))
    plt.bar(valid["method"], valid["silhouette"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Silhouette")
    plt.title("Сравнение методов кластеризации")
    plt.grid(True, axis="y")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


# =========================
# MAIN
# =========================
def main():
    ensure_dir(OUTPUT_DIR)

    df_raw, X_numeric, X_scaled, X_pca, pca, info = prepare_features(WIDE_CSV_PATH)

    print("Feature preparation:")
    print(info)

    # для удобства добавим PCA-координаты к файлам
    df_files = df_raw[["file_name"]].copy()
    df_files["pca1"] = X_pca[:, 0]
    df_files["pca2"] = X_pca[:, 1] if X_pca.shape[1] > 1 else 0.0

    all_grid_results = []
    all_best_results = []

    # методы с заданным k
    for method_name in methods_with_fixed_k():
        print(f"\n=== {method_name} ===")
        grid_df, best = evaluate_method_over_k(method_name, X_pca, K_CANDIDATES)
        all_grid_results.append(grid_df)

        if best is not None:
            all_best_results.append({
                "method": method_name,
                "best_k": best["best_k"],
                "silhouette": best["silhouette"],
                "calinski_harabasz": best["calinski_harabasz"],
                "davies_bouldin": best["davies_bouldin"],
                "n_clusters_found": best["n_clusters_found"],
                "n_noise": best["n_noise"],
            })

            labels_df = df_files.copy()
            labels_df["cluster"] = best["labels"]
            labels_df.to_csv(os.path.join(OUTPUT_DIR, f"{method_name}_labels.csv"), index=False)

            save_method_scatter(df_files, best["labels"], method_name, OUTPUT_DIR)

            print(f"best_k={best['best_k']}, silhouette={best['silhouette']:.4f}")

    # методы без заранее заданного k
    for method_name in methods_without_k():
        print(f"\n=== {method_name} ===")
        row_df, result = evaluate_density_method(method_name, X_pca)
        all_grid_results.append(row_df)

        if result is not None:
            all_best_results.append({
                "method": method_name,
                "best_k": np.nan,
                "silhouette": result["silhouette"],
                "calinski_harabasz": result["calinski_harabasz"],
                "davies_bouldin": result["davies_bouldin"],
                "n_clusters_found": result["n_clusters_found"],
                "n_noise": result["n_noise"],
            })

            labels_df = df_files.copy()
            labels_df["cluster"] = result["labels"]
            labels_df.to_csv(os.path.join(OUTPUT_DIR, f"{method_name}_labels.csv"), index=False)

            save_method_scatter(df_files, result["labels"], method_name, OUTPUT_DIR)

            print(
                f"clusters_found={result['n_clusters_found']}, "
                f"noise={result['n_noise']}, "
                f"silhouette={result['silhouette']}"
            )

    grid_results_df = pd.concat(all_grid_results, ignore_index=True)
    best_results_df = pd.DataFrame(all_best_results).sort_values("silhouette", ascending=False)

    grid_results_df.to_csv(os.path.join(OUTPUT_DIR, "all_methods_grid_results.csv"), index=False)
    best_results_df.to_csv(os.path.join(OUTPUT_DIR, "all_methods_best_results.csv"), index=False)

    save_summary_bar(
        best_results_df,
        os.path.join(OUTPUT_DIR, "methods_comparison_silhouette.png")
    )

    with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "all_methods_clustering_summary.xlsx"), engine="xlsxwriter") as writer:
        grid_results_df.to_excel(writer, sheet_name="GridResults", index=False)
        best_results_df.to_excel(writer, sheet_name="BestResults", index=False)
        df_files.to_excel(writer, sheet_name="PCAData", index=False)

    print("\n=== TOP METHODS ===")
    print(best_results_df.head(10))
    print("\nSaved to:", OUTPUT_DIR)


if __name__ == "__main__":
    main()

Feature preparation:
{'n_files': 525, 'n_features_final': 64, 'pca_components': 10}

=== KMeans ===
best_k=8, silhouette=0.4069

=== MiniBatchKMeans ===
best_k=8, silhouette=0.1631

=== Agglomerative_Ward ===
best_k=8, silhouette=0.4100

=== Agglomerative_Complete ===
best_k=8, silhouette=0.5575

=== Agglomerative_Average ===
best_k=8, silhouette=0.5538

=== Birch ===
best_k=8, silhouette=0.4329

=== SpectralClustering ===
best_k=8, silhouette=0.1677

=== GaussianMixture ===
best_k=8, silhouette=0.2639

=== DBSCAN ===
clusters_found=2, noise=235, silhouette=0.3752235661423656

=== OPTICS ===
clusters_found=2, noise=194, silhouette=0.5228903031174686

=== TOP METHODS ===
                   method  best_k  silhouette  calinski_harabasz  \
3  Agglomerative_Complete     8.0    0.557457         146.950427   
4   Agglomerative_Average     8.0    0.553841          26.652129   
9                  OPTICS     NaN    0.522890         102.548226   
5                   Birch     8.0    0.432947    

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


# =========================
# SETTINGS
# =========================
WIDE_CSV_PATH = '/content/data/wavelet_summary_wide.csv'
OUTPUT_DIR = '/content/data/nn_clustering_autoencoder'

USE_ONLY_COMMON_SCALES = True
COMMON_SCALES = ['A4', 'D4', 'D3', 'D2', 'D1']
DROP_META_COLS = ['file_name', 'wavelet', 'n_samples', 'max_level']
MAX_NAN_RATIO = 0.40

LATENT_DIM = 16
BATCH_SIZE = 64
EPOCHS = 100
LR = 1e-3
RANDOM_STATE = 42
K_CANDIDATES = [8]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


# =========================
# DATA PREP
# =========================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def is_common_scale_column(col_name, scales):
    return any(f'_{scale}_' in col_name for scale in scales)


def prepare_features(path):
    df = pd.read_csv(path)

    feature_cols = [c for c in df.columns if c not in DROP_META_COLS]
    if USE_ONLY_COMMON_SCALES:
        feature_cols = [c for c in feature_cols if is_common_scale_column(c, COMMON_SCALES)]

    X = df[feature_cols].copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce')

    nan_ratio = X.isna().mean()
    X = X.loc[:, nan_ratio <= MAX_NAN_RATIO]

    nunique = X.nunique(dropna=True)
    X = X.loc[:, nunique > 1]

    imputer = SimpleImputer(strategy='median')
    X_imp = imputer.fit_transform(X)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)

    return df, X_scaled, X.columns.tolist()


# =========================
# MODEL
# =========================
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z


def train_autoencoder(X_scaled):
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_tensor), batch_size=BATCH_SIZE, shuffle=True)

    model = Autoencoder(X_scaled.shape[1], LATENT_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()

    history = []

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0

        for (xb,) in loader:
            xb = xb.to(DEVICE)

            optimizer.zero_grad()
            x_hat, _ = model(xb)
            loss = criterion(x_hat, xb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * xb.size(0)

        epoch_loss = total_loss / len(X_tensor)
        history.append(epoch_loss)

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{EPOCHS} - loss: {epoch_loss:.6f}')

    return model, history


def get_latent(model, X_scaled):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(DEVICE)
        _, z = model(X_tensor)
    return z.cpu().numpy()


# =========================
# CLUSTERING
# =========================
def evaluate_kmeans(latent, k_values):
    rows = []
    best = None
    best_sil = -np.inf

    for k in k_values:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        labels = model.fit_predict(latent)

        sil = silhouette_score(latent, labels)
        ch = calinski_harabasz_score(latent, labels)
        db = davies_bouldin_score(latent, labels)

        rows.append({
            'k': k,
            'silhouette': sil,
            'calinski_harabasz': ch,
            'davies_bouldin': db
        })

        if sil > best_sil:
            best_sil = sil
            best = (k, labels, sil, ch, db)

    return pd.DataFrame(rows), best


def save_results(df_raw, latent, labels, history, scores_df, best_k):
    ensure_dir(OUTPUT_DIR)

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    latent_2d = pca.fit_transform(latent)

    result_df = df_raw[['file_name']].copy()
    result_df['cluster'] = labels
    result_df['z1'] = latent[:, 0]
    result_df['z2'] = latent[:, 1] if latent.shape[1] > 1 else 0.0
    result_df['pca1'] = latent_2d[:, 0]
    result_df['pca2'] = latent_2d[:, 1]

    result_df.to_csv(os.path.join(OUTPUT_DIR, 'ae_kmeans_labels.csv'), index=False)
    scores_df.to_csv(os.path.join(OUTPUT_DIR, 'ae_kmeans_scores.csv'), index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(history)
    plt.xlabel('Epoch')
    plt.ylabel('Reconstruction loss')
    plt.title('Autoencoder training loss')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'ae_loss.png'), dpi=150)
    plt.close()

    plt.figure(figsize=(8, 6))
    for cl in sorted(result_df['cluster'].unique()):
        part = result_df[result_df['cluster'] == cl]
        plt.scatter(part['pca1'], part['pca2'], label=f'cluster {cl}', s=20)
    plt.xlabel('PCA1')
    plt.ylabel('PCA2')
    plt.title(f'Autoencoder + KMeans (k={best_k})')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'ae_kmeans_pca.png'), dpi=150)
    plt.close()

    with pd.ExcelWriter(os.path.join(OUTPUT_DIR, 'ae_kmeans_summary.xlsx'), engine='xlsxwriter') as writer:
        result_df.to_excel(writer, sheet_name='Labels', index=False)
        scores_df.to_excel(writer, sheet_name='Scores', index=False)

    print('Saved:', OUTPUT_DIR)


def main():
    torch.manual_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)

    df_raw, X_scaled, feature_cols = prepare_features(WIDE_CSV_PATH)
    print('X shape:', X_scaled.shape)

    model, history = train_autoencoder(X_scaled)
    latent = get_latent(model, X_scaled)

    scores_df, best = evaluate_kmeans(latent, K_CANDIDATES)
    best_k, labels, sil, ch, db = best

    print(scores_df)
    print(f'Best k: {best_k}')
    print(f'Silhouette: {sil:.4f}')

    save_results(df_raw, latent, labels, history, scores_df, best_k)


if __name__ == '__main__':
    main()

X shape: (525, 64)
Epoch 10/100 - loss: 0.369090
Epoch 20/100 - loss: 0.179841
Epoch 30/100 - loss: 0.130909
Epoch 40/100 - loss: 0.103089
Epoch 50/100 - loss: 0.090146
Epoch 60/100 - loss: 0.081157
Epoch 70/100 - loss: 0.071903
Epoch 80/100 - loss: 0.063837
Epoch 90/100 - loss: 0.057122
Epoch 100/100 - loss: 0.053877
   k  silhouette  calinski_harabasz  davies_bouldin
0  8    0.459355         195.196228        0.955919
Best k: 8
Silhouette: 0.4594
Saved: /content/data/nn_clustering_autoencoder


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


WIDE_CSV_PATH = '/content/data/wavelet_summary_wide.csv'
OUTPUT_DIR = '/content/data/nn_clustering_vae'

USE_ONLY_COMMON_SCALES = True
COMMON_SCALES = ['A4', 'D4', 'D3', 'D2', 'D1']
DROP_META_COLS = ['file_name', 'wavelet', 'n_samples', 'max_level']
MAX_NAN_RATIO = 0.40

LATENT_DIM = 16
BATCH_SIZE = 64
EPOCHS = 120
LR = 1e-3
BETA = 1e-3
RANDOM_STATE = 42
K_CANDIDATES = [8]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def is_common_scale_column(col_name, scales):
    return any(f'_{scale}_' in col_name for scale in scales)


def prepare_features(path):
    df = pd.read_csv(path)

    feature_cols = [c for c in df.columns if c not in DROP_META_COLS]
    if USE_ONLY_COMMON_SCALES:
        feature_cols = [c for c in feature_cols if is_common_scale_column(c, COMMON_SCALES)]

    X = df[feature_cols].copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce')

    nan_ratio = X.isna().mean()
    X = X.loc[:, nan_ratio <= MAX_NAN_RATIO]

    nunique = X.nunique(dropna=True)
    X = X.loc[:, nunique > 1]

    imputer = SimpleImputer(strategy='median')
    X_imp = imputer.fit_transform(X)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)

    return df, X_scaled


class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()

        self.enc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
        )
        self.mu = nn.Linear(64, latent_dim)
        self.logvar = nn.Linear(64, latent_dim)

        self.dec = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def encode(self, x):
        h = self.enc(x)
        return self.mu(h), self.logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar, z


def vae_loss(x_hat, x, mu, logvar, beta):
    recon = nn.functional.mse_loss(x_hat, x, reduction='mean')
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kld, recon, kld


def train_vae(X_scaled):
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_tensor), batch_size=BATCH_SIZE, shuffle=True)

    model = VAE(X_scaled.shape[1], LATENT_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    history = []

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0

        for (xb,) in loader:
            xb = xb.to(DEVICE)

            optimizer.zero_grad()
            x_hat, mu, logvar, z = model(xb)
            loss, recon, kld = vae_loss(x_hat, xb, mu, logvar, BETA)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * xb.size(0)

        epoch_loss = total_loss / len(X_tensor)
        history.append(epoch_loss)

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{EPOCHS} - loss: {epoch_loss:.6f}')

    return model, history


def get_latent_mu(model, X_scaled):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(DEVICE)
        mu, logvar = model.encode(X_tensor)
    return mu.cpu().numpy()


def evaluate_kmeans(latent, k_values):
    rows = []
    best = None
    best_sil = -np.inf

    for k in k_values:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        labels = model.fit_predict(latent)

        sil = silhouette_score(latent, labels)
        ch = calinski_harabasz_score(latent, labels)
        db = davies_bouldin_score(latent, labels)

        rows.append({
            'k': k,
            'silhouette': sil,
            'calinski_harabasz': ch,
            'davies_bouldin': db
        })

        if sil > best_sil:
            best_sil = sil
            best = (k, labels, sil, ch, db)

    return pd.DataFrame(rows), best


def save_results(df_raw, latent, labels, history, scores_df, best_k):
    ensure_dir(OUTPUT_DIR)

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    latent_2d = pca.fit_transform(latent)

    result_df = df_raw[['file_name']].copy()
    result_df['cluster'] = labels
    result_df['z1'] = latent[:, 0]
    result_df['z2'] = latent[:, 1] if latent.shape[1] > 1 else 0.0
    result_df['pca1'] = latent_2d[:, 0]
    result_df['pca2'] = latent_2d[:, 1]

    result_df.to_csv(os.path.join(OUTPUT_DIR, 'vae_kmeans_labels.csv'), index=False)
    scores_df.to_csv(os.path.join(OUTPUT_DIR, 'vae_kmeans_scores.csv'), index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(history)
    plt.xlabel('Epoch')
    plt.ylabel('VAE loss')
    plt.title('VAE training loss')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'vae_loss.png'), dpi=150)
    plt.close()

    plt.figure(figsize=(8, 6))
    for cl in sorted(result_df['cluster'].unique()):
        part = result_df[result_df['cluster'] == cl]
        plt.scatter(part['pca1'], part['pca2'], label=f'cluster {cl}', s=20)
    plt.xlabel('PCA1')
    plt.ylabel('PCA2')
    plt.title(f'VAE + KMeans (k={best_k})')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'vae_kmeans_pca.png'), dpi=150)
    plt.close()

    print('Saved:', OUTPUT_DIR)


def main():
    torch.manual_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)

    df_raw, X_scaled = prepare_features(WIDE_CSV_PATH)

    model, history = train_vae(X_scaled)
    latent = get_latent_mu(model, X_scaled)

    scores_df, best = evaluate_kmeans(latent, K_CANDIDATES)
    best_k, labels, sil, ch, db = best

    print(scores_df)
    print(f'Best k: {best_k}')
    print(f'Silhouette: {sil:.4f}')

    save_results(df_raw, latent, labels, history, scores_df, best_k)


if __name__ == '__main__':
    main()

Epoch 10/120 - loss: 0.446215
Epoch 20/120 - loss: 0.221613
Epoch 30/120 - loss: 0.125919
Epoch 40/120 - loss: 0.106515
Epoch 50/120 - loss: 0.091147
Epoch 60/120 - loss: 0.081346
Epoch 70/120 - loss: 0.073239
Epoch 80/120 - loss: 0.069049
Epoch 90/120 - loss: 0.064615
Epoch 100/120 - loss: 0.060180
Epoch 110/120 - loss: 0.055306
Epoch 120/120 - loss: 0.055790
   k  silhouette  calinski_harabasz  davies_bouldin
0  8    0.338358         134.150986        0.970474
Best k: 8
Silhouette: 0.3384
Saved: /content/data/nn_clustering_vae


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency


CLUSTERS_CSV = '/content/data/clustering_results/file_clusters.csv'
OUT_DIR = '/content/data/cluster_metadata_analysis'


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def parse_filename(fname: str):
    base = os.path.basename(fname)
    base = re.sub(r'\.txt$', '', base, flags=re.IGNORECASE)

    # берём часть до первого пробела
    left = base.split(' ')[0].strip()

    patient = None
    experiment_code = None
    experiment_family = None
    experiment_num = None

    # всё до конечных цифр = patient, конечные цифры = experiment
    m = re.match(r'^(.*?)(\d+)$', left)

    if m:
        patient = m.group(1)
        experiment_num = int(m.group(2))
        experiment_code = str(experiment_num)

        # если всё же хочешь отдельное "семейство",
        # можно брать последние буквы в patient, если они есть
        fam_match = re.search(r'([A-Za-zА-Яа-яЁё]+)$', patient)
        experiment_family = fam_match.group(1) if fam_match else None
    else:
        patient = left

    return {
        'file_name': fname,
        'patient': patient,
        'experiment_code': experiment_code,
        'experiment_family': experiment_family,
        'experiment_num': experiment_num
    }

def cramers_v(confusion_matrix):
    chi2, p, dof, expected = chi2_contingency(confusion_matrix)
    n = confusion_matrix.to_numpy().sum()
    r, k = confusion_matrix.shape
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    denom = min((kcorr - 1), (rcorr - 1))
    v = np.sqrt(phi2corr / denom) if denom > 0 else np.nan
    return chi2, p, v


def save_heatmap(table, title, out_path):
    plt.figure(figsize=(12, 6))
    plt.imshow(table.values, aspect='auto')
    plt.xticks(range(table.shape[1]), table.columns, rotation=90)
    plt.yticks(range(table.shape[0]), table.index)
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def save_stacked_bar(prop_table, title, out_path):
    prop_table.plot(kind='bar', stacked=True, figsize=(12, 6))
    plt.title(title)
    plt.ylabel('proportion')
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def analyze_relation(df, category_col, out_dir):
    sub = df.dropna(subset=[category_col, 'cluster']).copy()

    count_table = pd.crosstab(sub[category_col], sub['cluster'])
    row_prop = pd.crosstab(sub[category_col], sub['cluster'], normalize='index')
    col_prop = pd.crosstab(sub[category_col], sub['cluster'], normalize='columns')

    chi2, p, v = cramers_v(count_table)

    count_table.to_csv(os.path.join(out_dir, f'{category_col}_vs_cluster_counts.csv'))
    row_prop.to_csv(os.path.join(out_dir, f'{category_col}_vs_cluster_row_prop.csv'))
    col_prop.to_csv(os.path.join(out_dir, f'{category_col}_vs_cluster_col_prop.csv'))

    save_heatmap(
        count_table,
        f'{category_col} vs cluster (counts)',
        os.path.join(out_dir, f'{category_col}_vs_cluster_heatmap.png')
    )
    save_stacked_bar(
        row_prop,
        f'{category_col}: cluster proportions',
        os.path.join(out_dir, f'{category_col}_vs_cluster_stacked.png')
    )

    stats_row = pd.DataFrame([{
        'category': category_col,
        'n_levels': sub[category_col].nunique(),
        'chi2_pvalue': p,
        'cramers_v': v
    }])

    return stats_row


def main():
    ensure_dir(OUT_DIR)

    clusters = pd.read_csv(CLUSTERS_CSV)

    meta = pd.DataFrame([parse_filename(x) for x in clusters['file_name']])
    df = clusters.merge(meta, on='file_name', how='left')

    df.to_csv(os.path.join(OUT_DIR, 'file_clusters_with_metadata.csv'), index=False)

    all_stats = []

    for col in ['patient', 'experiment_code', 'experiment_family']:
        stats_row = analyze_relation(df, col, OUT_DIR)
        all_stats.append(stats_row)

    stats_df = pd.concat(all_stats, ignore_index=True)
    stats_df.to_csv(os.path.join(OUT_DIR, 'cluster_metadata_relation_stats.csv'), index=False)

    print(df[['file_name', 'cluster', 'patient', 'experiment_code']].head())
    print('\nRelation stats:')
    print(stats_df)
    print('\nSaved to:', OUT_DIR)


if __name__ == '__main__':
    main()

                                     file_name  cluster      patient  \
0  BondarevaSI11 2024-05-14_14_40_37331603.txt        3  BondarevaSI   
1  BondarevaSI12 2024-05-14_14_41_21244788.txt        2  BondarevaSI   
2  BondarevaSI21 2024-05-14_14_46_43904177.txt        0  BondarevaSI   
3  BondarevaSI22 2024-05-14_14_47_16127144.txt        4  BondarevaSI   
4  BondarevaSI31 2024-05-14_14_49_59028654.txt        3  BondarevaSI   

  experiment_code  
0              11  
1              12  
2              21  
3              22  
4              31  

Relation stats:
            category  n_levels    chi2_pvalue  cramers_v
0            patient        28   1.533149e-06   0.158439
1    experiment_code        19  3.140469e-141   0.452035
2  experiment_family        28   1.533149e-06   0.158439

Saved to: /content/data/cluster_metadata_analysis


In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency


# =========================
# НАСТРОЙКИ
# =========================
METHOD_LABEL_FILES = [
    '/content/data/multi_clustering_results/KMeans_labels.csv',
    '/content/data/multi_clustering_results/MiniBatchKMeans_labels.csv',
    '/content/data/multi_clustering_results/Agglomerative_Ward_labels.csv',
    '/content/data/multi_clustering_results/Agglomerative_Complete_labels.csv',
    '/content/data/multi_clustering_results/Agglomerative_Average_labels.csv',
    '/content/data/multi_clustering_results/Birch_labels.csv',
    '/content/data/multi_clustering_results/SpectralClustering_labels.csv',
    '/content/data/multi_clustering_results/GaussianMixture_labels.csv',
    '/content/data/multi_clustering_results/DBSCAN_labels.csv',
    '/content/data/multi_clustering_results/OPTICS_labels.csv',
    '/content/data/nn_clustering_autoencoder/ae_kmeans_labels.csv',
    '/content/data/nn_clustering_vae/vae_kmeans_labels.csv',
    '/content/data/nn_clustering_dec/dec_labels.csv',
]

OUTPUT_DIR = '/content/data/all_methods_metadata_analysis'


# =========================
# ВСПОМОГАТЕЛЬНОЕ
# =========================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def parse_filename(fname: str):
    base = os.path.basename(fname)
    base = re.sub(r'\.txt$', '', base, flags=re.IGNORECASE)

    # берём часть до первого пробела
    left = base.split(' ')[0].strip()

    patient = None
    experiment_code = None
    experiment_family = None
    experiment_num = None

    # всё до конечных цифр = patient, конечные цифры = experiment
    m = re.match(r'^(.*?)(\d+)$', left)

    if m:
        patient = m.group(1)
        experiment_num = int(m.group(2))
        experiment_code = str(experiment_num)

        # если всё же хочешь отдельное "семейство",
        # можно брать последние буквы в patient, если они есть
        fam_match = re.search(r'([A-Za-zА-Яа-яЁё]+)$', patient)
        experiment_family = fam_match.group(1) if fam_match else None
    else:
        patient = left

    return {
        'file_name': fname,
        'patient': patient,
        'experiment_code': experiment_code,
        'experiment_family': experiment_family,
        'experiment_num': experiment_num
    }

def cramers_v(confusion_matrix):
    chi2, p, dof, expected = chi2_contingency(confusion_matrix)
    n = confusion_matrix.to_numpy().sum()
    r, k = confusion_matrix.shape
    phi2 = chi2 / n

    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    denom = min((kcorr - 1), (rcorr - 1))

    v = np.sqrt(phi2corr / denom) if denom > 0 else np.nan
    return chi2, p, v


def save_heatmap(table, title, out_path):
    plt.figure(figsize=(12, 6))
    plt.imshow(table.values, aspect='auto')
    plt.xticks(range(table.shape[1]), table.columns, rotation=90)
    plt.yticks(range(table.shape[0]), table.index)
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def save_stacked_bar(prop_table, title, out_path):
    prop_table.plot(kind='bar', stacked=True, figsize=(12, 6))
    plt.title(title)
    plt.ylabel('proportion')
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def analyze_relation(df, category_col, method_dir):
    sub = df.dropna(subset=[category_col, 'cluster']).copy()

    # если есть шумовые метки -1, их можно оставить или исключить
    # здесь оставим, чтобы видеть реальную структуру метода
    count_table = pd.crosstab(sub[category_col], sub['cluster'])
    row_prop = pd.crosstab(sub[category_col], sub['cluster'], normalize='index')
    col_prop = pd.crosstab(sub[category_col], sub['cluster'], normalize='columns')

    chi2, p, v = cramers_v(count_table)

    count_table.to_csv(os.path.join(method_dir, f'{category_col}_vs_cluster_counts.csv'))
    row_prop.to_csv(os.path.join(method_dir, f'{category_col}_vs_cluster_row_prop.csv'))
    col_prop.to_csv(os.path.join(method_dir, f'{category_col}_vs_cluster_col_prop.csv'))

    save_heatmap(
        count_table,
        f'{category_col} vs cluster (counts)',
        os.path.join(method_dir, f'{category_col}_vs_cluster_heatmap.png')
    )
    save_stacked_bar(
        row_prop,
        f'{category_col}: cluster proportions',
        os.path.join(method_dir, f'{category_col}_vs_cluster_stacked.png')
    )

    return {
        'category': category_col,
        'n_levels': sub[category_col].nunique(),
        'chi2_pvalue': p,
        'cramers_v': v
    }


def analyze_method(label_csv_path, output_root):
    method_name = os.path.splitext(os.path.basename(label_csv_path))[0]
    method_dir = os.path.join(output_root, method_name)
    ensure_dir(method_dir)

    df = pd.read_csv(label_csv_path)

    if 'file_name' not in df.columns or 'cluster' not in df.columns:
        print(f'SKIP: {label_csv_path} -> нет file_name/cluster')
        return None

    meta = pd.DataFrame([parse_filename(x) for x in df['file_name']])
    dfm = df.merge(meta, on='file_name', how='left')

    dfm.to_csv(os.path.join(method_dir, 'labels_with_metadata.csv'), index=False)

    stats_rows = []
    for col in ['patient', 'experiment_code', 'experiment_family']:
        try:
            row = analyze_relation(dfm, col, method_dir)
            row['method'] = method_name
            stats_rows.append(row)
        except Exception as e:
            stats_rows.append({
                'method': method_name,
                'category': col,
                'n_levels': np.nan,
                'chi2_pvalue': np.nan,
                'cramers_v': np.nan,
                'error': str(e)
            })

    stats_df = pd.DataFrame(stats_rows)
    stats_df.to_csv(os.path.join(method_dir, 'metadata_relation_stats.csv'), index=False)

    return stats_df


def main():
    ensure_dir(OUTPUT_DIR)

    all_stats = []

    for path in METHOD_LABEL_FILES:
        if not os.path.exists(path):
            print(f'NOT FOUND: {path}')
            continue

        print(f'Processing: {path}')
        stats_df = analyze_method(path, OUTPUT_DIR)
        if stats_df is not None:
            all_stats.append(stats_df)

    if not all_stats:
        print('Нет обработанных файлов.')
        return

    summary_df = pd.concat(all_stats, ignore_index=True)
    summary_df.to_csv(os.path.join(OUTPUT_DIR, 'all_methods_metadata_relation_summary.csv'), index=False)

    with pd.ExcelWriter(os.path.join(OUTPUT_DIR, 'all_methods_metadata_relation_summary.xlsx'), engine='xlsxwriter') as writer:
        summary_df.to_excel(writer, sheet_name='Summary', index=False)

    print('\n=== SUMMARY ===')
    print(summary_df.sort_values(['category', 'cramers_v'], ascending=[True, False]))
    print('\nSaved to:', OUTPUT_DIR)


if __name__ == '__main__':
    main()

Processing: /content/data/multi_clustering_results/KMeans_labels.csv
Processing: /content/data/multi_clustering_results/MiniBatchKMeans_labels.csv
Processing: /content/data/multi_clustering_results/Agglomerative_Ward_labels.csv
Processing: /content/data/multi_clustering_results/Agglomerative_Complete_labels.csv
Processing: /content/data/multi_clustering_results/Agglomerative_Average_labels.csv
Processing: /content/data/multi_clustering_results/Birch_labels.csv
Processing: /content/data/multi_clustering_results/SpectralClustering_labels.csv
Processing: /content/data/multi_clustering_results/GaussianMixture_labels.csv
Processing: /content/data/multi_clustering_results/DBSCAN_labels.csv
Processing: /content/data/multi_clustering_results/OPTICS_labels.csv
Processing: /content/data/nn_clustering_autoencoder/ae_kmeans_labels.csv
Processing: /content/data/nn_clustering_vae/vae_kmeans_labels.csv
NOT FOUND: /content/data/nn_clustering_dec/dec_labels.csv

=== SUMMARY ===
             category  n

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================
# НАСТРОЙКИ
# =========================
ROOT_DIR = '/content/data/all_methods_metadata_analysis'
OUTPUT_DIR = '/content/data/all_methods_metadata_visuals'

CATEGORIES = ['patient', 'experiment_code', 'experiment_family']

TOP_N_PATIENTS = 25
TOP_N_EXPERIMENT_CODES = 25
TOP_N_EXPERIMENT_FAMILIES = 25

DPI = 160


# =========================
# СЛУЖЕБНЫЕ ФУНКЦИИ
# =========================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def safe_read_csv(path, index_col=0):
    if os.path.exists(path):
        return pd.read_csv(path, index_col=index_col)
    return None


def maybe_reset_index_from_csv(df):
    """
    Иногда после чтения csv индекс уже корректный, иногда приходит как обычная колонка.
    """
    if df is None:
        return None
    if 'Unnamed: 0' in df.columns:
        df = df.rename(columns={'Unnamed: 0': 'index_col'}).set_index('index_col')
    return df


def get_top_n_for_category(category):
    if category == 'patient':
        return TOP_N_PATIENTS
    elif category == 'experiment_code':
        return TOP_N_EXPERIMENT_CODES
    elif category == 'experiment_family':
        return TOP_N_EXPERIMENT_FAMILIES
    return 25


def clip_top_rows(df, top_n=25):
    if df is None or df.empty:
        return df
    totals = df.sum(axis=1).sort_values(ascending=False)
    keep_idx = totals.head(top_n).index
    return df.loc[keep_idx]


def prettify_title(s):
    return s.replace('_', ' ').title()


def to_numeric_columns(df):
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    return out


def safe_neglog10(pvals):
    p = np.asarray(pvals, dtype=float)
    p = np.clip(p, 1e-300, 1.0)
    return -np.log10(p)


# =========================
# УНИВЕРСАЛЬНЫЕ ПЛОТЫ
# =========================
def save_heatmap(df, title, out_path, annotate=False):
    if df is None or df.empty:
        return

    data = df.values

    plt.figure(figsize=(max(8, 0.6 * df.shape[1] + 4), max(5, 0.35 * df.shape[0] + 2)))
    plt.imshow(data, aspect='auto')
    plt.colorbar()

    plt.xticks(range(df.shape[1]), df.columns, rotation=90)
    plt.yticks(range(df.shape[0]), df.index)

    if annotate and df.shape[0] <= 20 and df.shape[1] <= 10:
        for i in range(df.shape[0]):
            for j in range(df.shape[1]):
                val = df.iloc[i, j]
                if isinstance(val, (int, np.integer)):
                    txt = f'{val:d}'
                else:
                    txt = f'{val:.2f}'
                plt.text(j, i, txt, ha='center', va='center', fontsize=8)

    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=DPI)
    plt.close()


def save_stacked_bar(df, title, out_path, ylabel='proportion'):
    if df is None or df.empty:
        return

    plt.figure(figsize=(max(10, 0.5 * len(df.index)), 6))
    df.plot(kind='bar', stacked=True, ax=plt.gca())
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(out_path, dpi=DPI)
    plt.close()


def save_cluster_sizes_plot(labels_df, out_path, title):
    if labels_df is None or labels_df.empty:
        return

    counts = labels_df['cluster'].value_counts().sort_index()

    plt.figure(figsize=(8, 5))
    plt.bar(counts.index.astype(str), counts.values)
    plt.xlabel('cluster')
    plt.ylabel('n_files')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=DPI)
    plt.close()


def save_dominant_cluster_plot(dominant_df, title, out_path):
    if dominant_df is None or dominant_df.empty:
        return

    tmp = dominant_df.sort_values(['dominant_cluster_share', 'n_files'], ascending=[False, False]).head(25)

    plt.figure(figsize=(12, 6))
    plt.bar(tmp.index.astype(str), tmp['dominant_cluster_share'].values)
    plt.xticks(rotation=90)
    plt.ylim(0, 1.05)
    plt.ylabel('dominant cluster share')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=DPI)
    plt.close()


def save_top_levels_per_cluster(count_table, title_prefix, out_path_prefix, top_n=10):
    if count_table is None or count_table.empty:
        return

    clusters = list(count_table.columns)

    for cl in clusters:
        s = count_table[cl].sort_values(ascending=False).head(top_n)

        plt.figure(figsize=(10, 6))
        plt.barh(s.index.astype(str), s.values)
        plt.gca().invert_yaxis()
        plt.xlabel('count')
        plt.title(f'{title_prefix}: top {top_n} for cluster {cl}')
        plt.tight_layout()
        plt.savefig(f'{out_path_prefix}_cluster_{cl}.png', dpi=DPI)
        plt.close()


# =========================
# АНАЛИЗ DOMINANT CLUSTER
# =========================
def build_dominant_cluster_table(count_table, row_prop):
    if count_table is None or row_prop is None or count_table.empty or row_prop.empty:
        return None

    out = pd.DataFrame(index=count_table.index)
    out['n_files'] = count_table.sum(axis=1)
    out['dominant_cluster'] = row_prop.idxmax(axis=1)
    out['dominant_cluster_share'] = row_prop.max(axis=1)

    # добавим все доли по кластерам
    for c in row_prop.columns:
        out[f'cluster_{c}_share'] = row_prop[c]

    out = out.sort_values(['dominant_cluster_share', 'n_files'], ascending=[False, False])
    return out


# =========================
# ВИЗУАЛИЗАЦИЯ ПО ОТДЕЛЬНОМУ МЕТОДУ
# =========================
def visualize_method(method_dir, output_root):
    method_name = os.path.basename(method_dir)
    vis_dir = os.path.join(output_root, method_name)
    ensure_dir(vis_dir)

    labels_path = os.path.join(method_dir, 'labels_with_metadata.csv')
    if not os.path.exists(labels_path):
        print(f'SKIP {method_name}: no labels_with_metadata.csv')
        return None

    labels_df = pd.read_csv(labels_path)

    # 1. размеры кластеров
    save_cluster_sizes_plot(
        labels_df,
        os.path.join(vis_dir, 'cluster_sizes.png'),
        f'{method_name}: cluster sizes'
    )

    method_summary_rows = []

    # 2. по каждой категории
    for category in CATEGORIES:
        count_path = os.path.join(method_dir, f'{category}_vs_cluster_counts.csv')
        row_prop_path = os.path.join(method_dir, f'{category}_vs_cluster_row_prop.csv')
        col_prop_path = os.path.join(method_dir, f'{category}_vs_cluster_col_prop.csv')

        count_table = maybe_reset_index_from_csv(safe_read_csv(count_path, index_col=0))
        row_prop = maybe_reset_index_from_csv(safe_read_csv(row_prop_path, index_col=0))
        col_prop = maybe_reset_index_from_csv(safe_read_csv(col_prop_path, index_col=0))

        if count_table is None or row_prop is None:
            print(f'SKIP {method_name}/{category}: missing crosstab files')
            continue

        count_table = to_numeric_columns(count_table)
        row_prop = to_numeric_columns(row_prop)
        if col_prop is not None:
            col_prop = to_numeric_columns(col_prop)

        top_n = get_top_n_for_category(category)

        count_top = clip_top_rows(count_table, top_n)
        row_prop_top = clip_top_rows(row_prop, top_n)
        col_prop_top = clip_top_rows(col_prop, top_n) if col_prop is not None else None

        # counts heatmap
        save_heatmap(
            count_top,
            f'{method_name}: {category} vs cluster (counts, top {top_n})',
            os.path.join(vis_dir, f'{category}_counts_heatmap.png'),
            annotate=False
        )

        # row proportions heatmap
        save_heatmap(
            row_prop_top,
            f'{method_name}: {category} vs cluster (row proportions, top {top_n})',
            os.path.join(vis_dir, f'{category}_rowprop_heatmap.png'),
            annotate=False
        )

        # column proportions heatmap
        if col_prop_top is not None:
            save_heatmap(
                col_prop_top,
                f'{method_name}: {category} vs cluster (column proportions, top {top_n})',
                os.path.join(vis_dir, f'{category}_colprop_heatmap.png'),
                annotate=False
            )

        # stacked bar: row proportions
        save_stacked_bar(
            row_prop_top,
            f'{method_name}: {category} cluster distribution (top {top_n})',
            os.path.join(vis_dir, f'{category}_stacked_rowprop.png'),
            ylabel='row proportion'
        )

        # stacked bar: counts
        save_stacked_bar(
            count_top,
            f'{method_name}: {category} cluster counts (top {top_n})',
            os.path.join(vis_dir, f'{category}_stacked_counts.png'),
            ylabel='count'
        )

        # top levels per cluster
        save_top_levels_per_cluster(
            count_table,
            f'{method_name} - {category}',
            os.path.join(vis_dir, f'{category}_top_levels'),
            top_n=10
        )

        # dominant cluster table
        dominant_df = build_dominant_cluster_table(count_table, row_prop)
        if dominant_df is not None:
            dominant_df.to_csv(os.path.join(vis_dir, f'{category}_dominant_cluster.csv'))

            save_dominant_cluster_plot(
                dominant_df,
                f'{method_name}: dominant cluster purity for {category}',
                os.path.join(vis_dir, f'{category}_dominant_cluster_purity.png')
            )

            # краткая статистика
            method_summary_rows.append({
                'method': method_name,
                'category': category,
                'n_levels': len(dominant_df),
                'mean_dominant_share': dominant_df['dominant_cluster_share'].mean(),
                'median_dominant_share': dominant_df['dominant_cluster_share'].median(),
                'n_levels_with_purity_ge_0_8': (dominant_df['dominant_cluster_share'] >= 0.8).sum(),
                'n_levels_with_purity_ge_0_9': (dominant_df['dominant_cluster_share'] >= 0.9).sum(),
            })

    summary_df = pd.DataFrame(method_summary_rows)
    if not summary_df.empty:
        summary_df.to_csv(os.path.join(vis_dir, 'method_dominance_summary.csv'), index=False)

    return summary_df


# =========================
# ОБЩАЯ ВИЗУАЛИЗАЦИЯ ПО ВСЕМ МЕТОДАМ
# =========================
def save_grouped_bar_from_pivot(pivot_df, title, ylabel, out_path):
    if pivot_df is None or pivot_df.empty:
        return

    plt.figure(figsize=(max(10, 1.2 * len(pivot_df.index)), 6))
    x = np.arange(len(pivot_df.index))
    cols = list(pivot_df.columns)
    width = 0.8 / max(1, len(cols))

    for i, c in enumerate(cols):
        plt.bar(x + i * width, pivot_df[c].values, width=width, label=str(c))

    plt.xticks(x + width * (len(cols) - 1) / 2, pivot_df.index, rotation=90)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=DPI)
    plt.close()


def save_rank_plot(summary_df, category, value_col, out_path, ascending=False):
    sub = summary_df[summary_df['category'] == category].copy()
    if sub.empty:
        return

    sub = sub.sort_values(value_col, ascending=ascending)

    plt.figure(figsize=(10, 6))
    plt.barh(sub['method'], sub[value_col])
    plt.xlabel(value_col)
    plt.title(f'{category}: ranking by {value_col}')
    plt.tight_layout()
    plt.savefig(out_path, dpi=DPI)
    plt.close()


def visualize_global_summary(root_dir, output_dir):
    summary_path = os.path.join(root_dir, 'all_methods_metadata_relation_summary.csv')
    if not os.path.exists(summary_path):
        print('No all_methods_metadata_relation_summary.csv found')
        return

    summary_df = pd.read_csv(summary_path)

    # чистим
    if 'chi2_pvalue' in summary_df.columns:
        summary_df['neglog10_p'] = safe_neglog10(summary_df['chi2_pvalue'].fillna(1.0))
    else:
        summary_df['neglog10_p'] = np.nan

    # Cramer's V heatmap
    if 'cramers_v' in summary_df.columns:
        pivot_v = summary_df.pivot(index='method', columns='category', values='cramers_v')
        save_heatmap(
            pivot_v,
            'Methods × category: Cramér’s V',
            os.path.join(output_dir, 'summary_cramers_v_heatmap.png'),
            annotate=True
        )
        save_grouped_bar_from_pivot(
            pivot_v,
            'Cramér’s V by method and category',
            'Cramér’s V',
            os.path.join(output_dir, 'summary_cramers_v_groupedbar.png')
        )

        for cat in pivot_v.columns:
            save_rank_plot(
                summary_df,
                cat,
                'cramers_v',
                os.path.join(output_dir, f'rank_cramers_v_{cat}.png'),
                ascending=False
            )

    # -log10 p heatmap
    pivot_p = summary_df.pivot(index='method', columns='category', values='neglog10_p')
    save_heatmap(
        pivot_p,
        'Methods × category: -log10(p-value)',
        os.path.join(output_dir, 'summary_neglog10p_heatmap.png'),
        annotate=True
    )
    save_grouped_bar_from_pivot(
        pivot_p,
        '-log10(p-value) by method and category',
        '-log10(p)',
        os.path.join(output_dir, 'summary_neglog10p_groupedbar.png')
    )

    for cat in pivot_p.columns:
        save_rank_plot(
            summary_df,
            cat,
            'neglog10_p',
            os.path.join(output_dir, f'rank_neglog10p_{cat}.png'),
            ascending=False
        )

    # scatter: Cramér’s V vs -log10(p)
    if 'cramers_v' in summary_df.columns:
        plt.figure(figsize=(9, 7))
        for _, row in summary_df.iterrows():
            x = row['cramers_v']
            y = row['neglog10_p']
            label = f"{row['method']} | {row['category']}"
            plt.scatter(x, y)
            plt.text(x, y, label, fontsize=7)

        plt.xlabel("Cramér’s V")
        plt.ylabel("-log10(p-value)")
        plt.title("Association strength across methods/categories")
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'summary_scatter_cramers_vs_p.png'), dpi=DPI)
        plt.close()

    # сохраняем расширенную summary
    summary_df.to_csv(os.path.join(output_dir, 'all_methods_metadata_relation_summary_extended.csv'), index=False)

    # excel
    with pd.ExcelWriter(os.path.join(output_dir, 'all_methods_visual_summary.xlsx'), engine='xlsxwriter') as writer:
        summary_df.to_excel(writer, sheet_name='RelationSummary', index=False)
        if 'cramers_v' in summary_df.columns:
            pivot_v.to_excel(writer, sheet_name='Pivot_CramersV')
        pivot_p.to_excel(writer, sheet_name='Pivot_neglog10p')


# =========================
# ДОПОЛНИТЕЛЬНЫЙ СВОДНЫЙ АНАЛИЗ DOMINANCE
# =========================
def visualize_dominance_summaries(output_dir):
    method_dirs = [d for d in glob.glob(os.path.join(output_dir, '*')) if os.path.isdir(d)]

    all_rows = []
    for method_dir in method_dirs:
        path = os.path.join(method_dir, 'method_dominance_summary.csv')
        if os.path.exists(path):
            df = pd.read_csv(path)
            all_rows.append(df)

    if not all_rows:
        return

    dom_df = pd.concat(all_rows, ignore_index=True)
    dom_df.to_csv(os.path.join(output_dir, 'all_methods_dominance_summary.csv'), index=False)

    for metric in ['mean_dominant_share', 'median_dominant_share', 'n_levels_with_purity_ge_0_8', 'n_levels_with_purity_ge_0_9']:
        pivot = dom_df.pivot(index='method', columns='category', values=metric)
        save_heatmap(
            pivot,
            f'Dominance summary: {metric}',
            os.path.join(output_dir, f'dominance_{metric}_heatmap.png'),
            annotate=True
        )
        save_grouped_bar_from_pivot(
            pivot,
            f'Dominance summary: {metric}',
            metric,
            os.path.join(output_dir, f'dominance_{metric}_groupedbar.png')
        )

    with pd.ExcelWriter(os.path.join(output_dir, 'all_methods_dominance_summary.xlsx'), engine='xlsxwriter') as writer:
        dom_df.to_excel(writer, sheet_name='DominanceSummary', index=False)


# =========================
# MAIN
# =========================
def main():
    ensure_dir(OUTPUT_DIR)

    method_dirs = []
    for path in glob.glob(os.path.join(ROOT_DIR, '*')):
        if os.path.isdir(path):
            if os.path.exists(os.path.join(path, 'labels_with_metadata.csv')):
                method_dirs.append(path)

    all_method_summaries = []
    for method_dir in sorted(method_dirs):
        print(f'Visualizing method: {os.path.basename(method_dir)}')
        summary_df = visualize_method(method_dir, OUTPUT_DIR)
        if summary_df is not None and not summary_df.empty:
            all_method_summaries.append(summary_df)

    if all_method_summaries:
        merged = pd.concat(all_method_summaries, ignore_index=True)
        merged.to_csv(os.path.join(OUTPUT_DIR, 'all_method_dominance_summaries.csv'), index=False)

    visualize_global_summary(ROOT_DIR, OUTPUT_DIR)
    visualize_dominance_summaries(OUTPUT_DIR)

    print('\nDone.')
    print('All visualizations saved to:', OUTPUT_DIR)


if __name__ == '__main__':
    main()

Visualizing method: Agglomerative_Average_labels
Visualizing method: Agglomerative_Complete_labels
Visualizing method: Agglomerative_Ward_labels
Visualizing method: Birch_labels
Visualizing method: DBSCAN_labels
Visualizing method: GaussianMixture_labels
Visualizing method: KMeans_labels
Visualizing method: MiniBatchKMeans_labels
Visualizing method: OPTICS_labels
Visualizing method: SpectralClustering_labels
Visualizing method: ae_kmeans_labels
Visualizing method: vae_kmeans_labels

Done.
All visualizations saved to: /content/data/all_methods_metadata_visuals


In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd

from scipy.stats import binom, chisquare


# =========================
# НАСТРОЙКИ
# =========================
METHOD_LABEL_FILES = [
    '/content/data/multi_clustering_results/KMeans_labels.csv',
    '/content/data/multi_clustering_results/MiniBatchKMeans_labels.csv',
    '/content/data/multi_clustering_results/Agglomerative_Ward_labels.csv',
    '/content/data/multi_clustering_results/Agglomerative_Complete_labels.csv',
    '/content/data/multi_clustering_results/Agglomerative_Average_labels.csv',
    '/content/data/multi_clustering_results/Birch_labels.csv',
    '/content/data/multi_clustering_results/SpectralClustering_labels.csv',
    '/content/data/multi_clustering_results/GaussianMixture_labels.csv',
    '/content/data/multi_clustering_results/DBSCAN_labels.csv',
    '/content/data/multi_clustering_results/OPTICS_labels.csv',
    '/content/data/nn_clustering_autoencoder/ae_kmeans_labels.csv',
    '/content/data/nn_clustering_vae/vae_kmeans_labels.csv',
    '/content/data/nn_clustering_dec/dec_labels.csv',
]

OUTPUT_DIR = '/content/data/cluster_randomness_analysis'

ALPHA = 0.05
ANALYZE_COLUMNS = ['patient', 'experiment_code']


# =========================
# ВСПОМОГАТЕЛЬНОЕ
# =========================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def parse_filename(fname: str):
    """
    Примеры:
    oreshkin49 2024-04-14_10_35_36742659.txt
      -> patient=oreshkin, experiment=49

    BondarevaSI31 2024-05-14_14_49_59028654.txt
      -> patient=BondarevaSI, experiment=31
    """
    base = os.path.basename(fname)
    base = re.sub(r'\.txt$', '', base, flags=re.IGNORECASE)

    left = base.split(' ')[0].strip()

    patient = None
    experiment_code = None
    experiment_num = None

    m = re.match(r'^(.*?)(\d+)$', left)
    if m:
        patient = m.group(1)
        experiment_num = int(m.group(2))
        experiment_code = str(experiment_num)
    else:
        patient = left

    return {
        'file_name': fname,
        'patient': patient,
        'experiment_code': experiment_code,
        'experiment_num': experiment_num
    }


def normalize_counts_to_probs(counts_series):
    s = counts_series.sort_index()
    total = s.sum()
    if total == 0:
        return s * np.nan
    return s / total


def exact_binomial_interval(n, p, alpha=0.05):
    """
    Интервал допустимых значений числа успехов X при известном p:
    X ~ Bin(n, p)
    """
    low, high = binom.interval(1 - alpha, n, p)
    return int(low), int(high)


def build_group_result(group_name, n_group, cluster_labels, global_probs, alpha=0.05):
    """
    cluster_labels: Series со значениями cluster только внутри группы
    global_probs: Series index=cluster, values=global probabilities
    """
    observed_counts = cluster_labels.value_counts().sort_index()
    observed_counts = observed_counts.reindex(global_probs.index, fill_value=0)

    observed_props = observed_counts / n_group
    expected_counts = global_probs * n_group

    per_cluster_rows = []
    all_inside = True

    for cl in global_probs.index:
        p = float(global_probs.loc[cl])
        obs = int(observed_counts.loc[cl])
        obs_prop = float(observed_props.loc[cl])

        low_count, high_count = exact_binomial_interval(n_group, p, alpha=alpha)
        low_prop = low_count / n_group
        high_prop = high_count / n_group

        inside = (low_count <= obs <= high_count)
        if not inside:
            all_inside = False

        per_cluster_rows.append({
            'cluster': cl,
            'global_prob': p,
            'observed_count': obs,
            'observed_prop': obs_prop,
            'expected_count': float(expected_counts.loc[cl]),
            'ci_low_count': low_count,
            'ci_high_count': high_count,
            'ci_low_prop': low_prop,
            'ci_high_prop': high_prop,
            'inside_ci': inside
        })

    # goodness-of-fit по всей группе сразу
    try:
        chi2_stat, chi2_p = chisquare(
            f_obs=observed_counts.values,
            f_exp=expected_counts.values
        )
    except Exception:
        chi2_stat, chi2_p = np.nan, np.nan

    group_random_like = bool(all_inside and (pd.isna(chi2_p) or chi2_p >= alpha))
    group_non_random = not group_random_like

    summary_row = {
        'group_name': group_name,
        'n_group': n_group,
        'chi2_stat': chi2_stat,
        'chi2_pvalue': chi2_p,
        'all_clusters_inside_ci': all_inside,
        'random_like': group_random_like,
        'non_random': group_non_random
    }

    details_df = pd.DataFrame(per_cluster_rows)
    return summary_row, details_df


def analyze_one_metadata(df, meta_col, alpha=0.05):
    """
    df должен содержать:
    file_name, cluster, meta_col
    """
    sub = df.dropna(subset=[meta_col, 'cluster']).copy()

    # для DBSCAN/OPTICS можно оставить -1 как отдельный кластер;
    # если хочешь исключить шум, раскомментируй:
    # sub = sub[sub['cluster'] != -1].copy()

    global_counts = sub['cluster'].value_counts().sort_index()
    global_probs = normalize_counts_to_probs(global_counts)

    summary_rows = []
    detail_rows = []

    for group_name, g in sub.groupby(meta_col):
        n_group = len(g)
        if n_group == 0:
            continue

        summary_row, details_df = build_group_result(
            group_name=group_name,
            n_group=n_group,
            cluster_labels=g['cluster'],
            global_probs=global_probs,
            alpha=alpha
        )

        summary_row[meta_col] = group_name
        summary_rows.append(summary_row)

        details_df[meta_col] = group_name
        detail_rows.append(details_df)

    summary_df = pd.DataFrame(summary_rows)
    detail_df = pd.concat(detail_rows, ignore_index=True) if detail_rows else pd.DataFrame()

    return global_probs, summary_df, detail_df


def analyze_method(label_csv_path, output_root, alpha=0.05):
    method_name = os.path.splitext(os.path.basename(label_csv_path))[0]
    method_dir = os.path.join(output_root, method_name)
    ensure_dir(method_dir)

    df = pd.read_csv(label_csv_path)

    if 'file_name' not in df.columns or 'cluster' not in df.columns:
        print(f'SKIP: {label_csv_path} -> no file_name/cluster')
        return None

    meta = pd.DataFrame([parse_filename(x) for x in df['file_name']])
    dfm = df.merge(meta, on='file_name', how='left')

    dfm.to_csv(os.path.join(method_dir, 'labels_with_metadata.csv'), index=False)

    method_summary_rows = []

    for meta_col in ANALYZE_COLUMNS:
        global_probs, summary_df, detail_df = analyze_one_metadata(dfm, meta_col, alpha=alpha)

        global_probs.to_csv(os.path.join(method_dir, f'{meta_col}_global_cluster_probs.csv'), header=['global_prob'])
        summary_df.to_csv(os.path.join(method_dir, f'{meta_col}_randomness_summary.csv'), index=False)
        detail_df.to_csv(os.path.join(method_dir, f'{meta_col}_randomness_details.csv'), index=False)

        if not summary_df.empty:
            summary_df['method'] = method_name
            summary_df['meta_type'] = meta_col
            method_summary_rows.append(summary_df)

    if method_summary_rows:
        method_summary_df = pd.concat(method_summary_rows, ignore_index=True)
        method_summary_df.to_csv(os.path.join(method_dir, 'method_randomness_summary.csv'), index=False)
        return method_summary_df

    return None


def main():
    ensure_dir(OUTPUT_DIR)

    all_method_rows = []

    for path in METHOD_LABEL_FILES:
        if not os.path.exists(path):
            print(f'NOT FOUND: {path}')
            continue

        print(f'Processing: {path}')
        res = analyze_method(path, OUTPUT_DIR, alpha=ALPHA)
        if res is not None:
            all_method_rows.append(res)

    if not all_method_rows:
        print('Нет результатов.')
        return

    final_df = pd.concat(all_method_rows, ignore_index=True)
    final_df.to_csv(os.path.join(OUTPUT_DIR, 'all_methods_randomness_summary.csv'), index=False)

    with pd.ExcelWriter(os.path.join(OUTPUT_DIR, 'all_methods_randomness_summary.xlsx'), engine='xlsxwriter') as writer:
        final_df.to_excel(writer, sheet_name='Summary', index=False)

    print('\n=== DONE ===')
    print(final_df.head(20))
    print('\nSaved to:', OUTPUT_DIR)


if __name__ == '__main__':
    main()

Processing: /content/data/multi_clustering_results/KMeans_labels.csv
Processing: /content/data/multi_clustering_results/MiniBatchKMeans_labels.csv
Processing: /content/data/multi_clustering_results/Agglomerative_Ward_labels.csv
Processing: /content/data/multi_clustering_results/Agglomerative_Complete_labels.csv
Processing: /content/data/multi_clustering_results/Agglomerative_Average_labels.csv
Processing: /content/data/multi_clustering_results/Birch_labels.csv
Processing: /content/data/multi_clustering_results/SpectralClustering_labels.csv
Processing: /content/data/multi_clustering_results/GaussianMixture_labels.csv
Processing: /content/data/multi_clustering_results/DBSCAN_labels.csv
Processing: /content/data/multi_clustering_results/OPTICS_labels.csv
Processing: /content/data/nn_clustering_autoencoder/ae_kmeans_labels.csv
Processing: /content/data/nn_clustering_vae/vae_kmeans_labels.csv
NOT FOUND: /content/data/nn_clustering_dec/dec_labels.csv

=== DONE ===
     group_name  n_group  c

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================
# НАСТРОЙКИ
# =========================
ROOT_DIR = '/content/data/cluster_randomness_analysis'
OUTPUT_DIR = '/content/data/cluster_randomness_visuals'

META_TYPES = ['patient', 'experiment_code']

TOP_N_GROUPS = 20
DPI = 160


# =========================
# ВСПОМОГАТЕЛЬНЫЕ
# =========================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def safe_read_csv(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    return None


def save_bar(values, labels, title, ylabel, out_path, rotation=45):
    plt.figure(figsize=(8, 5))
    plt.bar(labels, values)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotation, ha='right')
    plt.tight_layout()
    plt.savefig(out_path, dpi=DPI)
    plt.close()


def save_heatmap(df, title, out_path, cmap='viridis', annotate=False):
    if df is None or df.empty:
        return

    plt.figure(figsize=(max(8, 0.6 * df.shape[1] + 4), max(5, 0.35 * df.shape[0] + 2)))
    plt.imshow(df.values, aspect='auto', cmap=cmap)
    plt.colorbar()
    plt.xticks(range(df.shape[1]), df.columns, rotation=90)
    plt.yticks(range(df.shape[0]), df.index)

    if annotate and df.shape[0] <= 25 and df.shape[1] <= 12:
        for i in range(df.shape[0]):
            for j in range(df.shape[1]):
                val = df.iloc[i, j]
                if pd.notna(val):
                    plt.text(j, i, f'{val:.2f}' if isinstance(val, float) else str(val),
                             ha='center', va='center', fontsize=7)

    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=DPI)
    plt.close()


def safe_neglog10(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-300, 1.0)
    return -np.log10(p)


# =========================
# ГРАФИКИ ПО ОДНОМУ МЕТОДУ
# =========================
def plot_random_vs_nonrandom(summary_df, meta_type, out_dir, method_name):
    if summary_df is None or summary_df.empty:
        return

    counts = summary_df[['random_like', 'non_random']].sum()

    plt.figure(figsize=(6, 4))
    plt.bar(['random_like', 'non_random'], counts.values)
    plt.title(f'{method_name}: {meta_type} random vs non-random')
    plt.ylabel('n_groups')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'{meta_type}_random_vs_nonrandom.png'), dpi=DPI)
    plt.close()


def plot_top_nonrandom_groups(summary_df, meta_type, out_dir, method_name, top_n=20):
    if summary_df is None or summary_df.empty:
        return

    df = summary_df.copy()
    df['neglog10_p'] = safe_neglog10(df['chi2_pvalue'].fillna(1.0))
    df = df.sort_values('neglog10_p', ascending=False).head(top_n)

    x_col = 'group_name' if 'group_name' in df.columns else meta_type

    plt.figure(figsize=(12, 6))
    plt.bar(df[x_col].astype(str), df['neglog10_p'])
    plt.xticks(rotation=90)
    plt.ylabel('-log10(p-value)')
    plt.title(f'{method_name}: top {top_n} strongest deviations for {meta_type}')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'{meta_type}_top_nonrandom_groups.png'), dpi=DPI)
    plt.close()


def plot_ci_for_top_groups(details_df, summary_df, meta_type, out_dir, method_name, top_n=12):
    if details_df is None or details_df.empty or summary_df is None or summary_df.empty:
        return

    tmp = summary_df.copy()
    tmp['neglog10_p'] = safe_neglog10(tmp['chi2_pvalue'].fillna(1.0))
    tmp = tmp.sort_values('neglog10_p', ascending=False).head(top_n)

    group_col = meta_type if meta_type in details_df.columns else 'group_name'
    top_groups = tmp[group_col].astype(str).tolist()

    details_plot = details_df[details_df[group_col].astype(str).isin(top_groups)].copy()
    if details_plot.empty:
        return

    clusters = sorted(details_plot['cluster'].dropna().unique())
    groups = top_groups

    nrows = len(groups)
    fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=(10, max(3 * nrows, 6)), squeeze=False)

    for i, group in enumerate(groups):
        ax = axes[i, 0]
        gdf = details_plot[details_plot[group_col].astype(str) == group].copy()
        gdf = gdf.sort_values('cluster')

        x = np.arange(len(gdf))
        y = gdf['observed_prop'].values
        low = gdf['ci_low_prop'].values
        high = gdf['ci_high_prop'].values
        inside = gdf['inside_ci'].values

        for j in range(len(gdf)):
            ax.plot([x[j], x[j]], [low[j], high[j]], color='gray', linewidth=2)
            ax.scatter(
                x[j],
                y[j],
                color='green' if inside[j] else 'red',
                s=60,
                zorder=3
            )

        ax.set_xticks(x)
        ax.set_xticklabels([f'cluster {c}' for c in gdf['cluster']])
        ax.set_ylim(0, 1)
        ax.set_ylabel('proportion')
        ax.set_title(f'{group}')

    plt.suptitle(f'{method_name}: observed proportions vs CI ({meta_type})', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'{meta_type}_ci_top_groups.png'), dpi=DPI, bbox_inches='tight')
    plt.close()


def plot_inside_ci_heatmap(details_df, summary_df, meta_type, out_dir, method_name, top_n=25):
    if details_df is None or details_df.empty or summary_df is None or summary_df.empty:
        return

    tmp = summary_df.copy()
    tmp['neglog10_p'] = safe_neglog10(tmp['chi2_pvalue'].fillna(1.0))
    tmp = tmp.sort_values('neglog10_p', ascending=False).head(top_n)

    group_col = meta_type if meta_type in details_df.columns else 'group_name'
    top_groups = tmp[group_col].astype(str).tolist()

    d = details_df[details_df[group_col].astype(str).isin(top_groups)].copy()
    if d.empty:
        return

    d[group_col] = d[group_col].astype(str)
    pivot = d.pivot(index=group_col, columns='cluster', values='inside_ci')
    pivot = pivot.astype(float)

    save_heatmap(
        pivot,
        f'{method_name}: inside CI matrix ({meta_type})',
        os.path.join(out_dir, f'{meta_type}_inside_ci_heatmap.png'),
        cmap='RdYlGn',
        annotate=True
    )


def plot_observed_minus_expected_heatmap(details_df, summary_df, meta_type, out_dir, method_name, top_n=25):
    if details_df is None or details_df.empty or summary_df is None or summary_df.empty:
        return

    tmp = summary_df.copy()
    tmp['neglog10_p'] = safe_neglog10(tmp['chi2_pvalue'].fillna(1.0))
    tmp = tmp.sort_values('neglog10_p', ascending=False).head(top_n)

    group_col = meta_type if meta_type in details_df.columns else 'group_name'
    top_groups = tmp[group_col].astype(str).tolist()

    d = details_df[details_df[group_col].astype(str).isin(top_groups)].copy()
    if d.empty:
        return

    d[group_col] = d[group_col].astype(str)
    d['diff_prop'] = d['observed_prop'] - (d['expected_count'] / d['observed_count'].groupby(d[group_col]).transform('sum').replace(0, np.nan))

    # лучше считать через expected / n_group
    if 'n_group' in d.columns:
        d['expected_prop'] = d['expected_count'] / d['n_group']
    else:
        # fallback
        group_sizes = d.groupby(group_col)['observed_count'].transform('sum').replace(0, np.nan)
        d['expected_prop'] = d['expected_count'] / group_sizes

    d['diff_prop'] = d['observed_prop'] - d['expected_prop']

    pivot = d.pivot(index=group_col, columns='cluster', values='diff_prop')

    save_heatmap(
        pivot,
        f'{method_name}: observed - expected proportions ({meta_type})',
        os.path.join(out_dir, f'{meta_type}_obs_minus_exp_heatmap.png'),
        cmap='coolwarm',
        annotate=True
    )


def visualize_method(method_dir, output_root):
    method_name = os.path.basename(method_dir)
    vis_dir = os.path.join(output_root, method_name)
    ensure_dir(vis_dir)

    all_rows = []

    for meta_type in META_TYPES:
        summary_path = os.path.join(method_dir, f'{meta_type}_randomness_summary.csv')
        details_path = os.path.join(method_dir, f'{meta_type}_randomness_details.csv')

        summary_df = safe_read_csv(summary_path)
        details_df = safe_read_csv(details_path)

        if summary_df is None or details_df is None:
            print(f'SKIP {method_name}/{meta_type}: no randomness files')
            continue

        # добавим group_name при необходимости
        if 'group_name' not in summary_df.columns and meta_type in summary_df.columns:
            summary_df['group_name'] = summary_df[meta_type]

        if meta_type in details_df.columns:
            details_df['group_name'] = details_df[meta_type]

        # если в details нет n_group — добавим
        if 'n_group' not in details_df.columns and 'group_name' in details_df.columns:
            group_sizes = details_df.groupby('group_name')['observed_count'].sum().rename('n_group')
            details_df = details_df.merge(group_sizes, on='group_name', how='left')

        plot_random_vs_nonrandom(summary_df, meta_type, vis_dir, method_name)
        plot_top_nonrandom_groups(summary_df, meta_type, vis_dir, method_name, top_n=TOP_N_GROUPS)
        plot_ci_for_top_groups(details_df, summary_df, meta_type, vis_dir, method_name, top_n=min(12, TOP_N_GROUPS))
        plot_inside_ci_heatmap(details_df, summary_df, meta_type, vis_dir, method_name, top_n=TOP_N_GROUPS)
        plot_observed_minus_expected_heatmap(details_df, summary_df, meta_type, vis_dir, method_name, top_n=TOP_N_GROUPS)

        # summary per method/meta
        all_rows.append({
            'method': method_name,
            'meta_type': meta_type,
            'n_groups': len(summary_df),
            'n_random_like': int(summary_df['random_like'].sum()),
            'n_non_random': int(summary_df['non_random'].sum()),
            'share_non_random': float(summary_df['non_random'].mean()),
            'median_pvalue': float(summary_df['chi2_pvalue'].median()) if 'chi2_pvalue' in summary_df.columns else np.nan
        })

    if all_rows:
        method_summary = pd.DataFrame(all_rows)
        method_summary.to_csv(os.path.join(vis_dir, 'visual_method_summary.csv'), index=False)
        return method_summary

    return None


# =========================
# ОБЩАЯ ВИЗУАЛИЗАЦИЯ ПО ВСЕМ МЕТОДАМ
# =========================
def visualize_global_summary(root_dir, output_dir):
    method_dirs = [d for d in glob.glob(os.path.join(root_dir, '*')) if os.path.isdir(d)]

    rows = []
    for method_dir in method_dirs:
        method_name = os.path.basename(method_dir)
        for meta_type in META_TYPES:
            summary_path = os.path.join(method_dir, f'{meta_type}_randomness_summary.csv')
            summary_df = safe_read_csv(summary_path)
            if summary_df is None or summary_df.empty:
                continue

            rows.append({
                'method': method_name,
                'meta_type': meta_type,
                'n_groups': len(summary_df),
                'n_random_like': int(summary_df['random_like'].sum()),
                'n_non_random': int(summary_df['non_random'].sum()),
                'share_random_like': float(summary_df['random_like'].mean()),
                'share_non_random': float(summary_df['non_random'].mean()),
                'median_pvalue': float(summary_df['chi2_pvalue'].median()) if 'chi2_pvalue' in summary_df.columns else np.nan,
                'median_neglog10p': float(np.median(safe_neglog10(summary_df['chi2_pvalue'].fillna(1.0))))
            })

    if not rows:
        print('No global randomness summaries found')
        return

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(output_dir, 'all_methods_randomness_visual_summary.csv'), index=False)

    # share_non_random heatmap
    pivot_non_random = df.pivot(index='method', columns='meta_type', values='share_non_random')
    save_heatmap(
        pivot_non_random,
        'Share of non-random groups by method',
        os.path.join(output_dir, 'share_non_random_heatmap.png'),
        cmap='magma',
        annotate=True
    )

    # share_random_like heatmap
    pivot_random = df.pivot(index='method', columns='meta_type', values='share_random_like')
    save_heatmap(
        pivot_random,
        'Share of random-like groups by method',
        os.path.join(output_dir, 'share_random_like_heatmap.png'),
        cmap='viridis',
        annotate=True
    )

    # grouped bar: share_non_random
    methods = list(pivot_non_random.index)
    meta_cols = list(pivot_non_random.columns)
    x = np.arange(len(methods))
    width = 0.35 if len(meta_cols) <= 2 else 0.8 / len(meta_cols)

    plt.figure(figsize=(12, 6))
    for i, col in enumerate(meta_cols):
        plt.bar(x + i * width, pivot_non_random[col].values, width=width, label=col)

    plt.xticks(x + width * (len(meta_cols) - 1) / 2, methods, rotation=90)
    plt.ylabel('share_non_random')
    plt.title('Non-random share across methods')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'share_non_random_groupedbar.png'), dpi=DPI)
    plt.close()

    # grouped bar: median -log10p
    pivot_p = df.pivot(index='method', columns='meta_type', values='median_neglog10p')

    plt.figure(figsize=(12, 6))
    meta_cols = list(pivot_p.columns)
    for i, col in enumerate(meta_cols):
        plt.bar(x + i * width, pivot_p[col].values, width=width, label=col)

    plt.xticks(x + width * (len(meta_cols) - 1) / 2, methods, rotation=90)
    plt.ylabel('median -log10(p)')
    plt.title('Median significance of non-randomness across methods')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'median_neglog10p_groupedbar.png'), dpi=DPI)
    plt.close()

    with pd.ExcelWriter(os.path.join(output_dir, 'all_methods_randomness_visual_summary.xlsx'), engine='xlsxwriter') as writer:
        df.to_excel(writer, sheet_name='Summary', index=False)
        pivot_non_random.to_excel(writer, sheet_name='ShareNonRandom')
        pivot_random.to_excel(writer, sheet_name='ShareRandomLike')
        pivot_p.to_excel(writer, sheet_name='MedianNeglog10p')


# =========================
# MAIN
# =========================
def main():
    ensure_dir(OUTPUT_DIR)

    method_dirs = [d for d in glob.glob(os.path.join(ROOT_DIR, '*')) if os.path.isdir(d)]
    all_method_rows = []

    for method_dir in sorted(method_dirs):
        print(f'Visualizing: {os.path.basename(method_dir)}')
        summary = visualize_method(method_dir, OUTPUT_DIR)
        if summary is not None and not summary.empty:
            all_method_rows.append(summary)

    if all_method_rows:
        merged = pd.concat(all_method_rows, ignore_index=True)
        merged.to_csv(os.path.join(OUTPUT_DIR, 'visual_methods_summary.csv'), index=False)

    visualize_global_summary(ROOT_DIR, OUTPUT_DIR)

    print('\nDone.')
    print('Saved to:', OUTPUT_DIR)


if __name__ == '__main__':
    main()

Visualizing: Agglomerative_Average_labels
Visualizing: Agglomerative_Complete_labels
Visualizing: Agglomerative_Ward_labels
Visualizing: Birch_labels
Visualizing: DBSCAN_labels
Visualizing: GaussianMixture_labels
Visualizing: KMeans_labels
Visualizing: MiniBatchKMeans_labels
Visualizing: OPTICS_labels
Visualizing: SpectralClustering_labels
Visualizing: ae_kmeans_labels
Visualizing: vae_kmeans_labels

Done.
Saved to: /content/data/cluster_randomness_visuals


In [ ]:
import shutil
from google.colab import files

folder_path = '/content/data'   # папка, которую архивируем
archive_path = '/content/data8.zip'   # куда сохранить zip

shutil.make_archive(
    base_name=archive_path.replace('.zip', ''),
    format='zip',
    root_dir=folder_path
)

files.download(archive_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import glob
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

warnings.filterwarnings("ignore")

In [ ]:
RAW_LOGS_PATTERN = "/content/логи_все_вместе_2/*.txt"

OUTPUT_DIR = "/content/data/timeseries_regression_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42

# Сколько прошлых точек подаём модели
INPUT_WINDOW = 100

# Сколько будущих точек предсказываем
FORECAST_HORIZON = 10

# Шаг скользящего окна
STEP = 5

# Какие колонки датчиков использовать
SENSOR_COLS = []

for i in range(1, 5):
    SENSOR_COLS += [
        f"ax{i}", f"ay{i}", f"az{i}",
        f"gx{i}", f"gy{i}", f"gz{i}"
    ]

print(SENSOR_COLS)

['ax1', 'ay1', 'az1', 'gx1', 'gy1', 'gz1', 'ax2', 'ay2', 'az2', 'gx2', 'gy2', 'gz2', 'ax3', 'ay3', 'az3', 'gx3', 'gy3', 'gz3', 'ax4', 'ay4', 'az4', 'gx4', 'gy4', 'gz4']


In [ ]:
def load_sensor_file(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    header_idx = None

    for i, line in enumerate(lines):
        if line.lower().startswith("milliseconds;"):
            header_idx = i
            break

    if header_idx is None:
        raise ValueError(f"Не найден заголовок milliseconds; в файле: {path}")

    df = pd.read_csv(
        path,
        sep=";",
        skiprows=header_idx,
        engine="python",
        on_bad_lines="skip"
    )

    df = df.dropna(axis=1, how="all")

    return df

In [ ]:
def prepare_sensor_dataframe(path, sensor_cols):
    df = load_sensor_file(path)

    # приводим названия колонок к нормальному виду
    df.columns = [str(c).strip() for c in df.columns]

    missing_cols = [c for c in sensor_cols if c not in df.columns]

    if missing_cols:
        print(
            "WARNING:",
            os.path.basename(path),
            "missing columns:",
            missing_cols
        )

    # создаём таблицу строго с нужными колонками
    # если какой-то колонки нет — она будет заполнена NaN
    df = df.reindex(columns=sensor_cols)

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # заполняем пропуски
    df = df.interpolate(limit_direction="both")

    # если колонка полностью отсутствовала, interpolate её не заполнит
    # поэтому заполняем оставшиеся NaN нулями
    df = df.fillna(0)

    return df

In [ ]:
def make_windows_from_array(arr, input_window, forecast_horizon, step):
    X_list = []
    y_list = []

    n = len(arr)

    last_start = n - input_window - forecast_horizon

    for start in range(0, last_start + 1, step):
        x_window = arr[start:start + input_window]
        y_window = arr[
            start + input_window:
            start + input_window + forecast_horizon
        ]

        # Превращаем окно в одну строку признаков
        X_list.append(x_window.flatten())

        # Будущий фрагмент тоже превращаем в одну строку
        y_list.append(y_window.flatten())

    return np.array(X_list), np.array(y_list)

In [ ]:
INPUT_WINDOW = 100
FORECAST_HORIZON = 10

In [ ]:
def build_timeseries_dataset(raw_logs_pattern, sensor_cols, input_window, forecast_horizon, step):
    all_X = []
    all_y = []
    file_names = []

    paths = sorted(glob.glob(raw_logs_pattern))

    if not paths:
        raise FileNotFoundError(f"Не найдены файлы по шаблону: {raw_logs_pattern}")

    for path in paths:
        try:
            df = prepare_sensor_dataframe(path, sensor_cols)
            arr = df.values.astype(float)

            if len(arr) < input_window + forecast_horizon:
                print("SKIP short file:", os.path.basename(path), len(arr))
                continue

            X_file, y_file = make_windows_from_array(
                arr,
                input_window=input_window,
                forecast_horizon=forecast_horizon,
                step=step
            )

            all_X.append(X_file)
            all_y.append(y_file)
            file_names.extend([os.path.basename(path)] * len(X_file))

            print("OK:", os.path.basename(path), X_file.shape, y_file.shape)

        except Exception as e:
            print("SKIP:", os.path.basename(path), "->", e)

    if len(all_X) == 0:
        raise ValueError("Не удалось собрать датасет")

    X = np.vstack(all_X)
    y = np.vstack(all_y)

    return X, y, np.array(file_names)


X, y, file_names = build_timeseries_dataset(
    RAW_LOGS_PATTERN,
    SENSOR_COLS,
    INPUT_WINDOW,
    FORECAST_HORIZON,
    STEP
)

print("X:", X.shape)
print("y:", y.shape)

OK: BondarevaSI11 2024-05-14_14_40_37331603.txt (45, 2400) (45, 240)
OK: BondarevaSI12 2024-05-14_14_41_21244788.txt (62, 2400) (62, 240)
OK: BondarevaSI21 2024-05-14_14_46_43904177.txt (59, 2400) (59, 240)
OK: BondarevaSI22 2024-05-14_14_47_16127144.txt (76, 2400) (76, 240)
OK: BondarevaSI31 2024-05-14_14_49_59028654.txt (44, 2400) (44, 240)
OK: BondarevaSI32 2024-05-14_14_50_20916348.txt (49, 2400) (49, 240)
OK: BondarevaSI41 2024-05-14_15_05_04560811.txt (110, 2400) (110, 240)
OK: BondarevaSI42 2024-05-14_15_05_37721012.txt (179, 2400) (179, 240)
OK: BondarevaSI43 2024-05-14_15_06_17647229.txt (261, 2400) (261, 240)
OK: BondarevaSI44 2024-05-14_15_07_26862652.txt (153, 2400) (153, 240)
OK: BondarevaSI45 2024-05-14_15_07_58664325.txt (208, 2400) (208, 240)
OK: BondarevaSI46 2024-05-14_15_09_01968384.txt (329, 2400) (329, 240)
OK: BondarevaSI47 2024-05-14_15_10_01809501.txt (200, 2400) (200, 240)
OK: BondarevaSI48 2024-05-14_15_10_42176936.txt (238, 2400) (238, 240)
OK: BondarevaSI49 

In [ ]:
unique_files = np.unique(file_names)

train_files, test_files = train_test_split(
    unique_files,
    test_size=0.25,
    random_state=RANDOM_STATE
)

train_mask = np.isin(file_names, train_files)
test_mask = np.isin(file_names, test_files)

X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (79321, 2400) (79321, 240)
Test: (25319, 2400) (25319, 240)


In [ ]:
models_and_params = {
    "ridge": {
        "model": Ridge(),
        "params": {
            "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
        }
    },

    "elastic_net": {
        "model": ElasticNet(max_iter=5000, random_state=RANDOM_STATE),
        "params": {
            "model__alpha": [0.001, 0.01, 0.1, 1.0],
            "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
        }
    },

    "knn": {
        "model": KNeighborsRegressor(),
        "params": {
            "model__n_neighbors": [3, 5, 7, 11],
            "model__weights": ["uniform", "distance"],
            "model__metric": ["euclidean", "manhattan"]
        }
    },

    "random_forest": {
        "model": RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "params": {
            "model__n_estimators": [100, 200, 400],
            "model__max_depth": [None, 10, 20, 40],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4]
        }
    },

    "extra_trees": {
        "model": ExtraTreesRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "params": {
            "model__n_estimators": [100, 200, 400],
            "model__max_depth": [None, 10, 20, 40],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4]
        }
    },

    "gradient_boosting": {
        "model": MultiOutputRegressor(
            GradientBoostingRegressor(random_state=RANDOM_STATE)
        ),
        "params": {
            "model__estimator__n_estimators": [50, 100, 200],
            "model__estimator__learning_rate": [0.01, 0.05, 0.1],
            "model__estimator__max_depth": [2, 3, 4]
        }
    },

    "mlp": {
        "model": MLPRegressor(
            max_iter=1000,
            early_stopping=True,
            random_state=RANDOM_STATE
        ),
        "params": {
            "model__hidden_layer_sizes": [
                (128,),
                (256,),
                (256, 128),
                (512, 256)
            ],
            "model__alpha": [1e-5, 1e-4, 1e-3],
            "model__learning_rate_init": [1e-4, 1e-3, 1e-2]
        }
    }
}

In [ ]:
def make_pipeline(model):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])


def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


results = []
best_model = None
best_model_name = None
best_rmse = np.inf

for name, cfg in models_and_params.items():
    print("\n" + "=" * 80)
    print("MODEL:", name)
    print("=" * 80)

    pipe = make_pipeline(cfg["model"])

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=cfg["params"],
        n_iter=10,
        scoring="neg_root_mean_squared_error",
        cv=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)

    y_pred = search.best_estimator_.predict(X_test)

    metrics = regression_metrics(y_test, y_pred)

    result = {
        "model": name,
        "best_cv_rmse": -search.best_score_,
        "test_MAE": metrics["MAE"],
        "test_RMSE": metrics["RMSE"],
        "test_R2": metrics["R2"],
        "best_params": search.best_params_
    }

    results.append(result)

    print("Best params:", search.best_params_)
    print("Test MAE:", metrics["MAE"])
    print("Test RMSE:", metrics["RMSE"])
    print("Test R2:", metrics["R2"])

    model_path = os.path.join(OUTPUT_DIR, f"{name}_model.joblib")
    joblib.dump(search.best_estimator_, model_path)

    if metrics["RMSE"] < best_rmse:
        best_rmse = metrics["RMSE"]
        best_model = search.best_estimator_
        best_model_name = name


results_df = pd.DataFrame(results).sort_values("test_RMSE")
results_df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"), index=False)

best_model_path = os.path.join(OUTPUT_DIR, "best_timeseries_model.joblib")
joblib.dump(best_model, best_model_path)

print("\nBEST MODEL:", best_model_name)
print("BEST RMSE:", best_rmse)
print("Saved:", best_model_path)

results_df

NameError: name 'np' is not defined

In [ ]:
config = {
    "input_window": INPUT_WINDOW,
    "forecast_horizon": FORECAST_HORIZON,
    "step": STEP,
    "sensor_cols": SENSOR_COLS,
    "best_model_name": best_model_name
}

config_path = os.path.join(OUTPUT_DIR, "config.json")

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=4)

print("Saved config:", config_path)

In [ ]:
def plot_prediction_example(y_true, y_pred, sensor_cols, forecast_horizon, sample_idx=0):
    n_channels = len(sensor_cols)

    true_seq = y_true[sample_idx].reshape(forecast_horizon, n_channels)
    pred_seq = y_pred[sample_idx].reshape(forecast_horizon, n_channels)

    for ch_idx, col in enumerate(sensor_cols):
        plt.figure(figsize=(8, 4))
        plt.plot(true_seq[:, ch_idx], label="true")
        plt.plot(pred_seq[:, ch_idx], label="predicted")
        plt.title(f"Prediction example: {col}")
        plt.xlabel("future step")
        plt.ylabel("value")
        plt.legend()
        plt.grid(True)
        plt.show()

In [ ]:
y_pred_best = best_model.predict(X_test)

plot_prediction_example(
    y_test,
    y_pred_best,
    SENSOR_COLS,
    FORECAST_HORIZON,
    sample_idx=0
)

In [ ]:
def predict_future_from_file(
    path,
    model_path=None,
    config_path=None
):
    if model_path is None:
        model_path = os.path.join(OUTPUT_DIR, "best_timeseries_model.joblib")

    if config_path is None:
        config_path = os.path.join(OUTPUT_DIR, "config.json")

    model = joblib.load(model_path)

    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)

    input_window = config["input_window"]
    forecast_horizon = config["forecast_horizon"]
    sensor_cols = config["sensor_cols"]

    df = prepare_sensor_dataframe(path, sensor_cols)
    arr = df.values.astype(float)

    if len(arr) < input_window:
        raise ValueError(
            f"В файле меньше точек, чем INPUT_WINDOW: {len(arr)} < {input_window}"
        )

    last_window = arr[-input_window:]

    X_new = last_window.flatten().reshape(1, -1)

    y_pred = model.predict(X_new)

    n_channels = len(sensor_cols)

    future = y_pred.reshape(forecast_horizon, n_channels)

    future_df = pd.DataFrame(future, columns=sensor_cols)

    return future_df